# AraBEST dialect experiments
### EECE 490 · Ali Zbeeb, Fatima Karnib, Haidar Madi

Start with **E11**, the variational information bottleneck study. It screens bottlenecks at 4K on ADI-20 development, carries the selected setting and matched controls to 1K and 16K, and measures transfer on full Casablanca validation.

1. Select **GPU T4 ×2** and enable **Internet**. GPU 0 trains the heads and GPU 1 encodes audio when both are visible. A one-GPU runtime runs the same protocol sequentially.
2. Optionally attach matching AraBEST caches to reuse completed extraction. The notebook reads checksum-verified full manifests from the pinned GitHub checkout. A historical full-input ZIP is also supported. Attach `adi20_8k_vs_16k_results.zip` only to reuse that comparison or its saved EXP-03 heads.
3. Use **Run All** in an interactive session if you want to access the individual ZIPs while work continues. A saved run also writes each ZIP as its stage finishes. Once the 4K screening inputs are ready, the encoder can prepare later E11 inputs while the heads train.

This public notebook generates balanced training subsets and an 800-clip Casablanca cohort from the pinned manifests. Use a local assignment file to replay exact historical subsets. Full-Casablanca E11 uses all 6,819 validation rows in either mode.

| Run preset | Settings cell |
|---|---|
| E11 only · default | `RUN = (11,)` |
| Paper experiments | `RUN = (2, 4, 11)` |
| All experiments | `RUN = (1, 2, 3, 4, 5, 6, 7, 11)` |

**Results arrive one experiment at a time.** Each stage writes its own ZIP and releases GPU memory. E11 also saves a ZIP after each training size. Set `PREFLIGHT_ONLY = True` to check inputs, devices, and cache coverage before starting.

<details>
<summary>Experiment index</summary>

| ID | Experiment |
|---|---|
| EXP-01 | Label efficiency across training sizes |
| EXP-02 | Layer selection and mixtures |
| EXP-03 | Sensitivity to audio perturbations |
| EXP-04 | Grouped corpus and dialect probes |
| EXP-05 | Supervised contrastive learning |
| EXP-06 | Domain-adversarial training |
| EXP-07 | Temporal pooling |
| E11 | Variational compression and transfer |

</details>

## Runtime and experiment settings

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
from zipfile import ZipFile

RUN = (11,)                    # See the launch guide for the other presets.
SIZES = (100, 200, 500, 1000, 2000, 4000, 8000, 16000, "full")
SEEDS = (490, 491, 492)
TRAIN_SIZE = 4000             # EXP-02, EXP-05, EXP-06
E11_EXTEND_SIZES = (1000, 16000)  # Run after the 4K development sweep.
E11_ENCODE_MISSING = True        # Fill only uncached required rows. False pauses on cache misses.
POOL_SIZE = 4000              # EXP-07 needs a separate frame cache.
UPDATES = 1000
CONVERGENCE_MAX_UPDATES = 5000
MAX_EXTRACT_HOURS = 6
ENCODE_BATCH = 4
PREFLIGHT_ONLY = False        # Print input/cache coverage, then stop before extraction.
SOURCE_DOMAINS_CSV = ""       # Optional Kaggle path with clip_id,domain from verified ADI training metadata.
FULL_PACKAGE = "ADI_Acoustic_Full_Training.zip"
SAVED_8K_PACKAGE = "ADI20_Casablanca_Fusion_Ready_Inputs.zip"
SCALING_RESULTS_PACKAGE = "adi20_8k_vs_16k_results.zip"
PERTURB_HEAD_SIZE = "auto"    # "auto" prefers saved 16K, 8000 forces 8K, 16000 requires the 16K archive.
COHORT_ASSIGNMENTS_FILE = ""  # Optional path to a local, exact cohort assignment JSON.
FIXED_DRAW = 490              # Fixed evaluations and augmentation RNG.
USE_SEPARATE_ENCODER_GPU = True    # Auto-enable cuda:1 with two visible GPUs.
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
CACHE_ROOT = WORK / "adi20_research_cache"
FEATURE_INPUT_ROOTS = (INPUT,)
REPO = Path("/tmp/adi-research-repo")
COMMIT = "b64437a37e7eda2f1721a56860278640bf085bdd"
REPO_URL = "https://github.com/alizbeeb5/Arabic-Dialect-Identification.git"

assert RUN and set(RUN) <= {1, 2, 3, 4, 5, 6, 7, 11}
assert len(set(E11_EXTEND_SIZES)) == len(E11_EXTEND_SIZES) and set(E11_EXTEND_SIZES) <= {1000, 16000}
assert UPDATES > 0 and CONVERGENCE_MAX_UPDATES >= UPDATES
assert MAX_EXTRACT_HOURS > 0 and ENCODE_BATCH > 0
assert PERTURB_HEAD_SIZE in ("auto", 8000, 16000)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

def install_missing(packages):
    needed = [pkg for name, pkg in packages.items() if importlib.util.find_spec(name) is None]
    if needed:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)

def setup():
    WORK.mkdir(parents=True, exist_ok=True)
    REPO.mkdir(parents=True, exist_ok=True)
    def git(*args, **kwargs):
        return subprocess.run(["git", "-C", str(REPO), *args], **kwargs)
    if not (REPO / ".git").exists():
        git("init", "--quiet", check=True)
    if git("cat-file", "-e", f"{COMMIT}^{{commit}}", stderr=subprocess.DEVNULL).returncode:
        git("fetch", "--quiet", "--depth=1", REPO_URL, COMMIT, check=True)
    git("checkout", "--quiet", "--detach", COMMIT, check=True)
    sys.path.insert(0, str(REPO))
    install_missing({
        "soundfile": "soundfile", "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
    })

setup()

## 2. Pin the training and evaluation cohorts

Scaling subsets are nested within each seed. The 2K, 4K, 8K and 16K anchors are fixed across seeds, and smaller arms are balanced draws by seed. The default creates new anchors deterministically. Use an exact local assignment file to replay the historical membership. The full split retains its original class frequencies. The source train/dev boundary and evaluation clip IDs are verified before training.

When its archive is attached, the notebook can import the historical 8K/16K comparison on full Casablanca validation (6,819 clips). Its metrics use the original 20-way head outputs.

ADI-20 is evaluated on the original 200-clip 20-class pilot and an 800-clip shared-dialect cohort sampled from full dev. Supporting transfer experiments use a balanced 800-clip Casablanca cohort. E11 uses full Casablanca validation. On shared-dialect evaluations, the notebook reports macro-F1 over the eight reference dialects with both unrestricted 20-way predictions and an eight-way restricted argmax.

In [ ]:
import gc
import hashlib
import io
import json
import math
import shutil
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.nn import functional as F

from src.data import freeze_splits as fs
from src.data.analyze_casablanca_grouping import parse_original_path
from src.experiments import acoustic_classifier as clf, layer_mixture as lm
from src.models import acoustic_encoder as ae

torch.set_num_threads(2)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DUAL_GPU_ACTIVE = USE_SEPARATE_ENCODER_GPU and torch.cuda.device_count() >= 2
ENCODER_DEVICE = torch.device("cuda:1") if DUAL_GPU_ACTIVE else DEVICE
A100_40GB = (DEVICE.type == "cuda" and "A100" in torch.cuda.get_device_name(DEVICE)
             and torch.cuda.get_device_properties(DEVICE).total_memory >= 38 * 1024**3)
MANIFEST_DIR = REPO / "data/manifests"
CONFIG = json.loads((REPO / "configs/layer_mixture.json").read_text())
OPT = CONFIG["optimization"]
LABELS = json.loads((REPO / "configs/labels.json").read_text())["label_order"]
LABEL_ID = {label: i for i, label in enumerate(LABELS)}
SHARED = tuple(sorted(LABEL_ID[x] for x in ("ALG", "EGY", "JOR", "MAU", "MOR", "PAL", "UAE", "YEM")))
SELECTED = CONFIG["mixtures"]["mixture_middle"]
FULL_HASHES = {
    "train": "56a75aa929e2a2c4592012954386cb3d91ff7f256a7004ebe7c4a776573ea4ba",
    "dev": "525ecea4b39c504b28858ddbfaf0a954af703c91ae8edddb252a0a6f2c8d138b",
}
EVAL_HASHES = {
    "dev_pilot": "fb8684c35ef68e25834d5a4c82cddbd31044a899a63b665c41d3f9a179052d02",
    "casablanca_validation": "5ed7a36491ba6a41782302e1f3d31c3033b4d72ac18f1f7a2d04e85177dd064e",
}
ADI_SOURCE = "UBC-NLP/NADI_2026_ADI20_micro"
CAS_SOURCE = "UBC-NLP/Casablanca"
DATA_REVISIONS = {
    ADI_SOURCE: "c8da454c9f474bb075d3fe608940ee29e8ccfa43",
    CAS_SOURCE: "8951b1b88e28c1107142ced57967b8d16350951d",
}
TAG = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
OUT = WORK / f"adi20_research_{TAG}"
OUT.mkdir(parents=True)
FUSION_ZIP = next(iter(sorted(INPUT.rglob(SAVED_8K_PACKAGE))), None)
SCALING_SOURCE = next(iter(sorted(INPUT.rglob(SCALING_RESULTS_PACKAGE))), None)
if SCALING_SOURCE is None:
    folders = sorted(p.parent for p in INPUT.rglob("train_16000")
                     if (p / "zscore_stats.npz").is_file() and (p.parent / "protocol.json").is_file()
                     and (p.parent / "results.csv").is_file())
    SCALING_SOURCE = next(iter(folders), None)

def read_jsonl(blob, digest):
    if hashlib.sha256(blob).hexdigest() != digest:
        raise ValueError("Manifest checksum mismatch. Attach the original project bundle.")
    return [json.loads(line) for line in blob.splitlines() if line]

def full_manifests():
    package = next(iter(sorted(INPUT.rglob(FULL_PACKAGE))), None)
    extracted = [p.parent for p in INPUT.rglob("train.jsonl") if p.parent.name == "full"]
    if package:
        with ZipFile(package) as archive:
            return tuple(read_jsonl(archive.read(f"ADI_Acoustic_Full_Training/data/manifests/full/{name}.jsonl"),
                                    digest) for name, digest in FULL_HASHES.items())
    for folder in sorted(extracted):
        if (folder / "dev.jsonl").is_file():
            return tuple(read_jsonl((folder / f"{name}.jsonl").read_bytes(), digest)
                         for name, digest in FULL_HASHES.items())
    folder = REPO / "src/data/manifests"
    return tuple(read_jsonl((folder / f"{name}.jsonl").read_bytes(), digest)
                 for name, digest in FULL_HASHES.items())

def pinned_manifest(name):
    return read_jsonl((MANIFEST_DIR / f"{name}.jsonl").read_bytes(), EVAL_HASHES[name])

def y_of(rows):
    return np.array([LABEL_ID[r["label"]] for r in rows], dtype=np.int64)

def usable(row):
    return row.get("preparation_status", "ok") == "ok"

def unique(rows):
    return list({r["clip_id"]: r for r in rows}.values())

def digest_rows(rows):
    ids = "\n".join(r["clip_id"] for r in rows)
    return hashlib.sha256(ids.encode()).hexdigest()

def cas(row):
    return row["source_dataset"] == CAS_SOURCE

def recording_groups(rows):
    rows = unique(rows)
    keys = fs.Groups([r["clip_id"] for r in rows])
    seen = {}
    known = set()
    for row in rows:
        cid, dataset = row["clip_id"], row["source_dataset"]
        parsed = parse_original_path(row.get("original_path")) if cas(row) else None
        episode = parsed["episode_file"] if parsed and parsed["seg_id"] == row.get("seg_id") else None
        for field, value in (
            ("speaker", (dataset, row["speaker_id"]) if row.get("speaker_id") else None),
            ("source", row.get("source_id")),
            ("episode", episode),
            ("encoded", row.get("audio_sha256")),
            ("pcm", row.get("pcm_sha256")),
        ):
            if not value:
                continue
            if field in {"speaker", "source", "episode"}:
                known.add(cid)
            keys.union(seen.setdefault((field, value), cid), cid)
    return {r["clip_id"]: keys.find(r["clip_id"]) for r in rows}, known

def stratified_sample(rows, labels, per_class, seed, spread=True):
    if spread:
        groups, _ = recording_groups(rows)
        quotas = {LABELS[i]: per_class for i in labels}
        picked, _, _ = fs.sample_balanced(rows, groups, list(quotas), quotas, seed)
    else:
        rng = np.random.default_rng(seed)
        buckets = defaultdict(list)
        for row in rows:
            buckets[row["label"]].append(row)
        picked = [buckets[LABELS[i]][j] for i in labels
                  for j in rng.permutation(len(buckets[LABELS[i]]))[:per_class]]
    if Counter(r["label"] for r in picked) != {LABELS[i]: per_class for i in labels}:
        raise ValueError(f"Need {per_class} clips in each of {len(labels)} requested dialects.")
    return picked

def subset(n, seed):
    if n == "full":
        return TRAIN
    if n in ANCHORS:
        return ANCHORS[n]
    if not isinstance(n, int) or n < len(LABELS) or n % len(LABELS) or n > max(ANCHORS):
        raise ValueError("Choose 'full' or a size ≤16000 divisible by 20.")
    upper = min(k for k in ANCHORS if k > n)
    lower = max((k for k in ANCHORS if k < n), default=0)
    existing = ANCHORS.get(lower, [])
    ids = {r["clip_id"] for r in existing}
    candidates = [r for r in ANCHORS[upper] if r["clip_id"] not in ids]
    return existing + stratified_sample(candidates, range(20), (n - lower) // 20, seed)

def rows_for(labels, rows):
    allowed = {LABELS[i] for i in labels}
    return [r for r in rows if r["label"] in allowed]

def grouped_holdout(rows, strata, want_fit, want_test, seed, groups):
    strata = np.asarray(strata)
    codes, inverse = np.unique(strata, return_inverse=True)
    group = np.array([groups[r["clip_id"]] for r in rows])
    splitter = GroupShuffleSplit(n_splits=128, test_size=.27, random_state=seed)
    best = None
    for fit, test in splitter.split(inverse, groups=group):
        fit_min = int(np.bincount(inverse[fit], minlength=len(codes)).min())
        test_min = int(np.bincount(inverse[test], minlength=len(codes)).min())
        score = (min(fit_min / want_fit, test_min / want_test, 1),
                 min(fit_min, want_fit) + min(test_min, want_test))
        if min(fit_min, test_min) > 1 and (best is None or score > best[0]):
            best = (score, fit, test, min(fit_min, want_fit), min(test_min, want_test))
            if score[0] == 1:
                break
    if best is None:
        raise ValueError("Unable to create a group-disjoint holdout with support in all strata.")
    _, fit, test, fit_quota, test_quota = best
    rng = np.random.default_rng(seed)
    def balance(positions, quota):
        return [rows[i] for code in codes for i in rng.permutation(positions[strata[positions] == code])[:quota]]
    return balance(fit, fit_quota), balance(test, test_quota)

def check_disjoint(fit_rows, test_rows, groups):
    def ids(rows):
        return {groups[r["clip_id"]] for r in rows}
    if ids(fit_rows) & ids(test_rows):
        raise ValueError("Fitting and evaluation intersect by clip, speaker, recording or exact audio hash.")

In [ ]:
ANCHOR_SIZES = (2000, 4000, 8000, 16000)

def rows_at_positions(rows, positions, expected_size):
    if (not isinstance(positions, list) or len(positions) != expected_size
            or any(type(p) is not int or not 0 <= p < len(rows) for p in positions)
            or len(set(positions)) != expected_size):
        raise ValueError(f"Expected {expected_size} unique, zero-based manifest row positions.")
    return [rows[p] for p in positions]

def make_cohorts(train, casablanca, groups):
    if COHORT_ASSIGNMENTS_FILE:
        with Path(COHORT_ASSIGNMENTS_FILE).open() as handle:
            assignments = json.load(handle)
        anchors = {size: rows_at_positions(train, assignments["train"][str(size)], size)
                   for size in ANCHOR_SIZES}
        balanced = rows_at_positions(casablanca, assignments["casablanca_balanced"], 800)
        return anchors, balanced

    labels = range(len(LABELS))
    seed_sample = stratified_sample(train, labels, max(ANCHOR_SIZES) // len(LABELS), FIXED_DRAW)
    buckets = defaultdict(list)
    for row in seed_sample:
        buckets[row["label"]].append(row)
    anchors = {
        size: [row for label in LABELS for row in buckets[label][:size // len(LABELS)]]
        for size in ANCHOR_SIZES
    }
    train_groups = {groups[row["clip_id"]] for row in train}
    eligible = [row for row in casablanca if groups[row["clip_id"]] not in train_groups]
    balanced = stratified_sample(rows_for(SHARED, eligible), SHARED, 100, FIXED_DRAW)
    return anchors, balanced


In [ ]:
TRAIN, DEV_FULL = full_manifests()
DEV20 = pinned_manifest("dev_pilot")
CAS_ALL = pinned_manifest("casablanca_validation")
GROUPS, KNOWN_GROUPS = recording_groups(TRAIN + DEV_FULL + CAS_ALL)
ANCHORS, CAS800 = make_cohorts(TRAIN, CAS_ALL, GROUPS)
for expected_size, rows in ANCHORS.items():
    assert len(rows) == expected_size and len(unique(rows)) == expected_size
    assert Counter(y_of(rows)) == dict.fromkeys(range(20), expected_size // 20)
for small, large in zip(sorted(ANCHORS), sorted(ANCHORS)[1:]):
    assert {r["clip_id"] for r in ANCHORS[small]} <= {r["clip_id"] for r in ANCHORS[large]}
assert len(TRAIN) == 67015 and len(DEV_FULL) == 10806
assert len(DEV20) == 200 and len(CAS800) == 800 and Counter(y_of(CAS800)) == dict.fromkeys(SHARED, 100)
assert {r["clip_id"] for r in DEV20} <= {r["clip_id"] for r in DEV_FULL}
DEV8 = stratified_sample(rows_for(SHARED, DEV_FULL), SHARED, 100, FIXED_DRAW)
check_disjoint(TRAIN, DEV20 + DEV8 + CAS800, GROUPS)
EVAL = {"ADI20_dev200": DEV20, "ADI20_shared800": DEV8, "Casablanca_fixed800": CAS800}

PROBE_SPLITS = {}
if 4 in RUN:
    probe_rows = [r for r in unique(DEV8 + CAS800) if usable(r)]
    # Each stratum represents a dialect and corpus jointly.
    strata = [f"{r['label']}:{int(cas(r))}" for r in probe_rows]
    for seed in SEEDS:
        fit, test = grouped_holdout(probe_rows, strata, 48, 16, seed, GROUPS)
        check_disjoint(fit, test, GROUPS)
        PROBE_SPLITS[seed] = (fit, test)

SOURCE_DOMAINS = {}
ADAPT_TRAIN, ADAPT_TEST = [], []
if 6 in RUN and SOURCE_DOMAINS_CSV:
    domain_table = pd.read_csv(SOURCE_DOMAINS_CSV, dtype=str).dropna(subset=["clip_id", "domain"])
    if domain_table["clip_id"].duplicated().any():
        raise ValueError("Each clip_id needs exactly one verified source domain.")
    SOURCE_DOMAINS = dict(zip(domain_table["clip_id"], domain_table["domain"].str.strip()))
    SOURCE_DOMAINS = {k: v for k, v in SOURCE_DOMAINS.items() if v and v.lower() != "unknown"}
    if not SOURCE_DOMAINS:
        raise ValueError("The supplied source-domain table has no usable labels.")
elif 6 in RUN:
    # Dedicated, fixed split within Casablanca validation. Groups never cross this boundary.
    train_groups = {GROUPS[r["clip_id"]] for r in TRAIN + DEV20 + DEV8}
    available = [r for r in CAS_ALL if usable(r) and GROUPS[r["clip_id"]] not in train_groups]
    ADAPT_TRAIN, ADAPT_TEST = grouped_holdout(available, [r["label"] for r in available],
                                             100, 75, FIXED_DRAW + 17, GROUPS)
    check_disjoint(ADAPT_TRAIN, ADAPT_TEST, GROUPS)
    check_disjoint(TRAIN, ADAPT_TEST, GROUPS)
    print(f"EXP-06 supervised adaptation: {len(ADAPT_TRAIN)} fit / {len(ADAPT_TEST)} held-out target clips.")

# Save clip assignments before extraction. Membership is independent of model results.
def save_assignment(name, rows):
    pd.DataFrame([{
        "clip_id": r["clip_id"], "dialect": r["label"],
        "corpus": "Casablanca" if cas(r) else "ADI20",
        "group_id": GROUPS[r["clip_id"]], "group_from_metadata": r["clip_id"] in KNOWN_GROUPS,
    } for r in rows]).to_csv(OUT / f"{name}.csv.gz", index=False)

for name, rows in EVAL.items():
    save_assignment(name, rows)
for seed, (fit, test) in PROBE_SPLITS.items():
    save_assignment(f"exp04_{seed}_fit", fit)
    save_assignment(f"exp04_{seed}_test", test)
if ADAPT_TRAIN:
    save_assignment("exp06_target_fit", ADAPT_TRAIN)
    save_assignment("exp06_target_holdout", ADAPT_TEST)

print({name: dict(Counter(r["label"] for r in rows)) for name, rows in EVAL.items()})

## 3. Reuse or extract AraBEST features

The notebook verifies cache provenance and an audio-content key per clip. Pooled rows retain all 26 FP32 representations. Extraction timings measure the transformation and encoder pass for newly encoded clips. EXP-07 stores selected-layer frame sequences in FP16 with variable lengths. EXP-03 retains a separate pooled cache for each transform.

A recorded audio failure counts as a missed prediction and appears in coverage. Unexpected fetch, checksum, decoding or inference errors stop the affected run. Completed cache parts survive a wall-clock pause. An empty matching cache means the corresponding clips will be encoded from the pinned Hugging Face source.

In [ ]:
import soundfile as sf

FEATURE_CONFIG = clf.feature_config(CONFIG["encoder"])
PROVENANCE = json.loads(json.dumps(lm.expected_provenance(FEATURE_CONFIG)))
LAYERS = PROVENANCE["layer_names"]
WIDTH = PROVENANCE["output_dim"]
LEGACY_NAME = PROVENANCE["feature_config_sha256"][:16]
FEATURE_DEADLINE = time.monotonic() + MAX_EXTRACT_HOURS * 3600
PERTURBATIONS = {
    "pitch_minus2": ("pitch", -2), "pitch_plus2": ("pitch", 2),
    "rate_085": ("rate", .85), "rate_115": ("rate", 1.15),
    "shuffle_250ms": ("shuffle", .25), "shuffle_500ms": ("shuffle", .50),
    "telephone": ("bandpass", None), "noise_20db": ("noise", 20),
    "noise_10db": ("noise", 10), "volume_rms": ("volume", None),
}

def mounted_configs():
    configs = sorted({config for root in FEATURE_INPUT_ROOTS for config in root.rglob("config.json")})
    # The earlier fusion bundle also carries a small pooled-cache shard.
    if FUSION_ZIP:
        from tempfile import mkdtemp
        with ZipFile(FUSION_ZIP) as archive:
            cache_members = [item for item in archive.namelist() if item.startswith("features/")]
            if cache_members:
                temp = Path(mkdtemp(prefix="adi-legacy-features-"))
                for member in cache_members:
                    path = Path(member)
                    if ".." in path.parts or path.is_absolute():
                        raise ValueError("Unexpected feature path in the historical bundle.")
                    archive.extract(member, temp)
                configs.extend(sorted(temp.rglob("config.json")))
    return configs

MOUNTED_CONFIGS = mounted_configs()

class ExtractionPaused(Exception):
    pass

def atomic_text(path, value):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(value, encoding="utf-8")
    tmp.replace(path)

class FeatureBank:
    def __init__(self, name, frames=False):
        self.frames = frames
        self.name = name
        spec = {"encoder": PROVENANCE, "mode": name, "frames": frames, "format": 1}
        if frames:
            spec["selected_layers"] = list(SELECTED)
        if name in PERTURBATIONS:
            spec.update(transform=list(PERTURBATIONS[name]), seed=FIXED_DRAW)
        self.spec_hash = hashlib.sha256(ae.canonical_json(spec)).hexdigest()[:20]
        self.root = CACHE_ROOT / name / self.spec_hash
        self.index = {}
        # Fresh parts use separate index sidecars. The old Kaggle outputs have one index.jsonl.
        active_config = self.root / "config.json"
        configs = (active_config, *MOUNTED_CONFIGS) if active_config.is_file() else MOUNTED_CONFIGS
        for config in configs:
            if config.parent.name not in {LEGACY_NAME, self.spec_hash}:
                continue
            stored = json.loads(config.read_text())
            if config.parent.name == LEGACY_NAME:
                if name != "mean" or any(stored.get(k) != PROVENANCE[k]
                                         for k in ae.FeatureCache.IDENTITY_FIELDS):
                    continue
            elif stored != spec:
                continue
            index_paths = sorted(config.parent.glob("part-*.index.jsonl"))
            if config.with_name("index.jsonl").is_file():
                index_paths.insert(0, config.with_name("index.jsonl"))
            part_names = {part.name for part in config.parent.glob("part-*.npz")}
            for path in index_paths:
                for line in path.read_text().splitlines():
                    item = json.loads(line)
                    part = config.parent / item["part"]
                    if part.name in part_names:
                        self.index.setdefault(item["clip_id"], (part, item))
        self.root.mkdir(parents=True, exist_ok=True)
        atomic_text(self.root / "config.json", json.dumps(spec))

    def key(self, row):
        return ae.clip_cache_key(row["clip_id"], row["audio_sha256"], PROVENANCE)

    def has(self, row):
        entry = self.index.get(row["clip_id"])
        return bool(entry and row.get("audio_sha256") and entry[1]["cache_key"] == self.key(row))

    def missing(self, rows):
        return [r for r in rows if usable(r) and not self.has(r)]

    def write(self, rows, values, seconds):
        ids = [r["clip_id"] for r in rows]
        keys = [self.key(r) for r in rows]
        suffix = hashlib.sha256("".join(keys).encode()).hexdigest()[:24]
        stem = f"part-{suffix}"
        part = self.root / f"{stem}.npz"
        temp = self.root / f"{stem}.tmp"
        with temp.open("wb") as stream:
            if self.frames:
                offsets = np.r_[0, np.cumsum([len(value) for value in values])]
                np.savez(stream, clip_ids=ids, cache_keys=keys, features=np.concatenate(values), offsets=offsets)
            else:
                np.savez(stream, clip_ids=ids, cache_keys=keys, features=np.stack(values))
        temp.replace(part)
        items = [{"clip_id": cid, "part": part.name, "row": i, "cache_key": key,
                  "seconds": seconds / len(ids), "session": TAG}
                 for i, (cid, key) in enumerate(zip(ids, keys))]
        atomic_text(self.root / f"{stem}.index.jsonl", "".join(json.dumps(x) + "\n" for x in items))
        self.index.update({item["clip_id"]: (part, item) for item in items})

    def load(self, rows):
        # Caller preserves invalid evaluation rows separately, so array order is explicit here.
        result = [None] * len(rows) if self.frames else np.empty((len(rows), len(LAYERS), WIDTH), np.float32)
        parts = defaultdict(list)
        for position, row in enumerate(rows):
            if not self.has(row):
                raise ValueError(f"Cache '{self.name}' is missing {row['clip_id']}.")
            part, item = self.index[row["clip_id"]]
            parts[part].append((position, item))
        for part, positions in parts.items():
            with np.load(part, allow_pickle=False) as data:
                values, ids, keys = data["features"], data["clip_ids"], data["cache_keys"]
                if not np.isfinite(values).all():
                    raise ValueError(f"Nonfinite cached features in {part}.")
                offsets = data["offsets"] if self.frames else None
                for position, item in positions:
                    i = item["row"]
                    if ids[i] != item["clip_id"] or keys[i] != item["cache_key"]:
                        raise ValueError(f"Wrong feature row in {part}")
                    result[position] = values[offsets[i]:offsets[i+1]].copy() if self.frames else values[i]
        return result

    def resources(self, rows):
        current = [self.index[r["clip_id"]][1] for r in unique(rows) if self.has(r)]
        return {
            "feature_seconds_this_session": sum(x["seconds"] for x in current if x.get("session") == TAG),
            "feature_clips_new": sum(x.get("session") == TAG for x in current),
            "feature_clips_reused": sum(x.get("session") != TAG for x in current),
        }

MEAN = FeatureBank("mean")
FRAMES = FeatureBank("frames", frames=True) if 7 in RUN else None
AUGMENTED = {name: FeatureBank(name) for name in PERTURBATIONS} if 3 in RUN else {}
ENCODER = None

def get_encoder():
    global ENCODER
    if ENCODER is None:
        if ENCODER_DEVICE.type != "cuda" and not ALLOW_CPU_ENCODING:
            raise RuntimeError("Enable a Kaggle GPU for audio extraction, or attach a complete compatible cache.")
        token = hf_token()
        if token:
            os.environ["HF_TOKEN"] = token
        try:
            installed = version("speechbrain")
        except PackageNotFoundError:
            installed = ""
        if installed != "1.1.2":
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "speechbrain==1.1.2"],
                           check=True)
        install_missing({"hyperpyyaml": "hyperpyyaml", "sentencepiece": "sentencepiece",
                         "joblib": "joblib", "tqdm": "tqdm"})
        ENCODER = ae.AcousticEncoder.from_pretrained(FEATURE_CONFIG, device=str(ENCODER_DEVICE), cache_dir="/tmp/adi-hf")
        if json.loads(json.dumps(ENCODER.provenance())) != PROVENANCE:
            raise ValueError("The loaded encoder differs from the cache recipe.")
    return ENCODER

ALLOW_CPU_ENCODING = False

def wave_from_bytes(row, payload):
    if not payload or hashlib.sha256(payload).hexdigest() != row["audio_sha256"]:
        raise ValueError(f"Audio hash mismatch for {row['clip_id']}. Check the pinned dataset or WAV mirror.")
    wave, rate = sf.read(io.BytesIO(payload), dtype="float32", always_2d=True)
    return ae.prepare_waveform(wave.mean(axis=1), rate, FEATURE_CONFIG)

def perturb(wave, row, name):
    from scipy.signal import butter, sosfilt

    family, value = PERTURBATIONS[name]
    audio = wave.numpy()
    rate = FEATURE_CONFIG.sample_rate_hz
    seed = int(hashlib.sha256(f"{FIXED_DRAW}:{family}:{row['clip_id']}".encode()).hexdigest()[:16], 16)
    rng = np.random.default_rng(seed)
    if family in {"pitch", "rate"}:
        install_missing({"librosa": "librosa"})
        import librosa
        if family == "pitch":
            audio = librosa.effects.pitch_shift(audio, sr=rate, n_steps=value)
        else:
            audio = librosa.effects.time_stretch(audio, rate=value)
    elif family == "shuffle":
        width = round(rate * value)
        count = len(audio) // width
        if count > 1:
            head = audio[:count * width].reshape(count, width)
            audio = np.concatenate([head[rng.permutation(count)].reshape(-1), audio[count * width:]])
    elif family == "bandpass":
        audio = sosfilt(butter(6, (300, 3400), btype="bandpass", fs=rate, output="sos"), audio)
    elif family == "noise":
        noise = rng.normal(size=len(audio))
        rms = np.linalg.norm(audio) / max(np.linalg.norm(noise), 1e-12)
        audio = audio + noise * rms * 10 ** (-value / 20)
    elif family == "volume":
        rms = max(float(np.sqrt(np.mean(audio ** 2))), 1e-12)
        peak = max(float(np.max(np.abs(audio))), 1e-12)
        audio = audio * min(.1 / rms, .99 / peak)
    return ae.prepare_waveform(np.asarray(audio, dtype=np.float32), rate, FEATURE_CONFIG)

def encoder_batch(waves, banks):
    with torch.inference_mode():
        encoder = get_encoder()
        hidden, mask, counts = encoder.encode(waves)
        result = {}
        if any(bank.frames for bank in banks):
            selected = hidden[SELECTED].mean(0)
            result["frames"] = [selected[i, :n].cpu().numpy().astype(np.float16) for i, n in enumerate(counts.tolist())]
        if any(not bank.frames for bank in banks):
            pooled = encoder.pool(hidden, mask).cpu().numpy()
            for bank in banks:
                if not bank.frames:
                    result[bank.name] = pooled
        if any(not np.isfinite(v).all() for batch in result.values() for v in batch):
            raise FloatingPointError("AraBEST returned nonfinite features.")
        return result

def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None

def parquet_waves(rows, stop=None):
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem, hf_hub_download
    from tempfile import TemporaryDirectory

    token = hf_token()
    if token:
        os.environ["HF_TOKEN"] = token
    hub = HfFileSystem(token=token)
    paths = defaultdict(list)
    for row in rows:
        source, rev, shard = (row[k] for k in ("source_dataset", "source_revision", "source_shard"))
        if DATA_REVISIONS.get(source) != rev or (source == CAS_SOURCE and "validation" not in shard):
            raise ValueError(f"Unexpected source shard for {row['clip_id']}")
        paths[source, rev, shard].append(row)
    for (source, rev, shard), members in sorted(paths.items()):
        if stop is not None and stop.is_set():
            return
        if time.monotonic() >= FEATURE_DEADLINE:
            raise ExtractionPaused
        print(f"{shard} · {len(members)} clips", flush=True)
        positions = {int(row["source_row"]): row for row in members}
        if len(positions) != len(members):
            raise ValueError(f"Duplicate source positions in {shard}")
        with TemporaryDirectory(prefix="adi-parquet-") as tmp:
            if source == CAS_SOURCE:
                local = hf_hub_download(source, shard, repo_type="dataset", revision=rev, token=token, local_dir=tmp)
                reader = open(local, "rb")
            else:
                reader = hub.open(f"datasets/{source}@{rev}/{shard}", "rb", block_size=8 * 1024**2)
            with reader:
                parquet = pq.ParquetFile(reader)
                offset = 0
                for group in range(parquet.num_row_groups):
                    stop = offset + parquet.metadata.row_group(group).num_rows
                    needed = sorted(i for i in positions if offset <= i < stop)
                    cursor = offset
                    if needed:
                        for batch in parquet.iter_batches(row_groups=[group], columns=["audio"], batch_size=32):
                            if stop is not None and stop.is_set():
                                return
                            if time.monotonic() >= FEATURE_DEADLINE:
                                raise ExtractionPaused
                            end = cursor + len(batch)
                            audio = batch.column(0).field("bytes")
                            block = [positions.pop(i) for i in needed if cursor <= i < end]
                            offsets = [row["source_row"] - cursor for row in block]
                            if block:
                                yield [(row, wave_from_bytes(row, audio[j].as_py())) for row, j in zip(block, offsets)]
                            cursor = end
                            if cursor > needed[-1]:
                                break
                    offset = stop
                    if not positions:
                        break
        if positions:
            raise ValueError(f"Missing manifest rows in {shard}.")

def ensure_features(rows, *banks, stop=None):
    rows = unique(rows)
    required = {r["clip_id"]: r for bank in banks for r in bank.missing(rows)}
    print(f"{', '.join(b.name for b in banks)} · {len(required):,}/{len(rows):,} clips need encoding", flush=True)
    if not required or (stop is not None and stop.is_set()):
        return
    if time.monotonic() >= FEATURE_DEADLINE:
        raise ExtractionPaused
    get_encoder()
    remaining = len(required)
    for block in parquet_waves(list(required.values()), stop):
        if stop is not None and stop.is_set():
            return
        pending = defaultdict(list)
        clean = [b for b in banks if b.name not in PERTURBATIONS]
        plans = ([clean] if clean else []) + [[b] for b in banks if b.name in PERTURBATIONS]
        for start in range(0, len(block), ENCODE_BATCH):
            batch = block[start:start+ENCODE_BATCH]
            for plan in plans:
                for group in {tuple(b for b in plan if not b.has(row)) for row, _ in batch} - {()}:
                    matching = [(r, w) for r, w in batch if tuple(b for b in plan if not b.has(r)) == group]
                    items, waves = map(list, zip(*matching))
                    tick = time.perf_counter()
                    if group[0].name in PERTURBATIONS:
                        waves = [perturb(w, r, group[0].name) for r, w in matching]
                    try:
                        values = encoder_batch(waves, group)
                    except torch.cuda.OutOfMemoryError:
                        with torch.cuda.device(ENCODER_DEVICE):
                            torch.cuda.empty_cache()
                        if len(waves) == 1:
                            raise
                        single = [encoder_batch([wave], group) for wave in waves]
                        values = {bank.name: [result[bank.name][0] for result in single] for bank in group}
                    elapsed = (time.perf_counter() - tick) / len(group)
                    for bank in group:
                        pending[bank].append((items, values[bank.name], elapsed))
        for bank, chunks in pending.items():
            items = [row for rows, _, _ in chunks for row in rows]
            values = [value for _, batch, _ in chunks for value in batch]
            bank.write(items, values, sum(elapsed for _, _, elapsed in chunks))
        remaining -= len(block)
        if remaining and time.monotonic() >= FEATURE_DEADLINE:
            raise ExtractionPaused

def require_train(rows, bank):
    bad = [r["clip_id"] for r in rows if not usable(r) or not bank.has(r)]
    if bad:
        raise ValueError(f"{bank.name}: {len(bad)} training rows are unavailable. First: {bad[0]}")

## 4. Shared training and scoring

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

REPORTS, PER_LABEL, RUN_STATUS = [], [], []

def readable(bank, rows):
    positions = [i for i, row in enumerate(rows) if usable(row) and bank.has(row)]
    return np.array(positions, dtype=int), [rows[i] for i in positions]

def standardizer(raw):
    total = np.zeros(raw.shape[1:], np.float64)
    squares = np.zeros_like(total)
    for start in range(0, len(raw), 256):
        block = raw[start:start+256].astype(np.float64)
        total += block.sum(0)
        squares += np.square(block).sum(0)
    mean = total / len(raw)
    sd = np.sqrt(np.maximum(squares / len(raw) - mean ** 2, 0))
    return mean.astype(np.float32), np.maximum(sd, CONFIG["preprocessing"]["std_floor"]).astype(np.float32)

def zscore(raw, stats, inplace=False):
    if not inplace:
        raw = raw.copy()
    raw -= stats[0]
    raw /= stats[1]
    return raw

def pool_mean(raw, layers=None):
    if layers is None:
        return raw.mean(axis=1)
    # Sum selected layers without creating a second full [N,L,D] tensor.
    return sum(raw[:, i] for i in layers) / len(layers)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def run_lr(x, y, seed):
    tick = time.perf_counter()
    model = LogisticRegression(C=1, max_iter=1000, random_state=seed)
    model.fit(x, y)
    elapsed = time.perf_counter() - tick
    iterations = int(model.n_iter_.max())
    return model, {"train_seconds": elapsed, "trainable_params": model.coef_.size + model.intercept_.size,
                   "updates": iterations, "fit_status": "solver_cap" if iterations == model.max_iter else "converged"}

def lr_scores(model, x, width):
    if not len(x):
        return np.empty((0, width))
    scores = model.decision_function(x)
    if scores.ndim == 1:
        scores = np.column_stack([np.zeros(len(scores)), scores])
    result = np.full((len(x), width), -np.inf)
    result[:, model.classes_] = scores
    return result

def collate(x, indices):
    if isinstance(x, torch.Tensor):
        positions = torch.as_tensor(indices, device=x.device)
        return (x.index_select(0, positions),)
    if isinstance(x, np.ndarray):
        return (torch.as_tensor(x[indices], device=DEVICE),)
    seq = [torch.from_numpy(x[i].astype(np.float32)) for i in indices]
    padded = nn.utils.rnn.pad_sequence(seq, batch_first=True).to(DEVICE)
    lengths = torch.tensor([len(a) for a in seq], device=DEVICE)
    mask = torch.arange(padded.shape[1], device=DEVICE)[None, :] < lengths[:, None]
    return padded, mask

@torch.inference_mode()
def torch_scores(model, x):
    model.eval()
    size = 128 if isinstance(x, np.ndarray) else 32
    blocks = [model(*collate(x, np.arange(start, min(start + size, len(x))))).cpu().numpy()
              for start in range(0, len(x), size)]
    return np.concatenate(blocks) if blocks else np.empty((0, len(LABELS)))

def mini_batches(n, batch_size, seed):
    rng = torch.Generator().manual_seed(seed)
    while True:
        for batch in torch.randperm(n, generator=rng).split(batch_size):
            yield batch.numpy()

def fit_torch(model, x, y, *, steps, seed, optimizer, batch_size=256, convergence=False,
              batches=None, objective=None):
    _, shuffle_seed = clf.run_seeds(seed)
    batches = mini_batches(len(y), batch_size, shuffle_seed) if batches is None else batches
    if objective is None:
        def objective(model, inputs, target, indices):
            return F.cross_entropy(model(*inputs), target)

    def snapshot(step, status):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        return ({k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                {"train_seconds": time.perf_counter() - started, "trainable_params": count_params(model),
                 "updates": step, "fit_status": status})

    convergence_cap = max(steps, CONVERGENCE_MAX_UPDATES) if convergence else steps
    monitor = np.random.default_rng(seed).choice(len(y), size=min(len(y), 2048), replace=False)
    interval = max(50, min(250, math.ceil(len(y) / batch_size)))
    training_labels = torch.as_tensor(y, device=DEVICE) if isinstance(x, torch.Tensor) else None
    started = time.perf_counter()
    best, stale, results = np.inf, 0, {}
    model.train()
    for step in range(1, convergence_cap + 1):
        indices = next(batches)
        if training_labels is None:
            target = torch.as_tensor(y[indices], device=DEVICE)
        else:
            target = training_labels.index_select(0, torch.as_tensor(indices, device=DEVICE))
        optimizer.zero_grad(set_to_none=True)
        loss = objective(model, collate(x, indices), target, indices)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), OPT["gradient_clip_norm"])
        optimizer.step()
        if step == steps:
            results["fixed"] = snapshot(step, "fixed")
        if convergence and "converged" not in results and step % interval == 0:
            scores = torch_scores(model, x[monitor])
            model.train()
            monitored = float(F.cross_entropy(torch.from_numpy(scores), torch.from_numpy(y[monitor])))
            if monitored < best - 1e-3:
                best, stale = monitored, 0
            else:
                stale += 1
            if stale == 4:
                results["converged"] = snapshot(step, "training_loss_plateau")
        if results.keys() >= ({"fixed", "converged"} if convergence else {"fixed"}):
            break
    if convergence and "converged" not in results:
        results["converged"] = snapshot(convergence_cap, "convergence_cap")
    model.eval()
    return results

def mixture(raw, labels, seed, steps=UPDATES, convergence=False):
    init, _ = clf.run_seeds(seed)
    model = lm.build_model(raw.shape, len(LABELS), init).to(DEVICE)
    snapshots = fit_torch(model, raw, labels, steps=steps, seed=seed, optimizer=lm.make_optimizer(model, OPT),
                          batch_size=OPT["batch_size"], convergence=convergence)
    return model, snapshots

def get_scores(bank, rows, predict):
    positions, valid = readable(bank, rows)
    logits = np.full((len(rows), len(LABELS)), np.nan)
    if valid:
        # Read the complete split once so every method's outputs keep the same clip order.
        logits[positions] = predict(bank.load(valid))
    return logits

def predictions_of(logits, allowed):
    pred = np.full(len(logits), -1, dtype=int)
    relevant = logits[:, allowed]
    valid = np.isfinite(relevant).any(1) & ~np.isnan(relevant).any(1) & ~np.isposinf(relevant).any(1)
    pred[valid] = np.asarray(allowed)[np.argmax(relevant[valid], axis=1)]
    return pred

def score_row(meta, split, mode, true, pred, refs, names):
    f1 = f1_score(true, pred, labels=list(refs), average=None, zero_division=0)
    key = {**meta, "split": split, "mode": mode}
    REPORTS.append({**key, "n": len(true), "scored": int((pred >= 0).sum()), "coverage": float((pred >= 0).mean()),
                    "accuracy": float((pred == true).mean()), "macro_f1": float(f1.mean())})
    for i, score in zip(refs, f1):
        PER_LABEL.append({**key, "dialect": names[i], "support": int((true == i).sum()), "f1": float(score)})

def report_dialects(exp, arm, seed, fit_rows, evals, bank, predictor, meta, *, checkpoint=None, preloaded=None):
    tag = f"exp{exp:02d}_{arm}_{seed}"
    pred_dir = OUT / "predictions"
    pred_dir.mkdir(exist_ok=True)
    score_meta = {"experiment": exp, "arm": arm, "seed": seed, "train_n": len(fit_rows),
                  "train_ids_sha256": digest_rows(fit_rows), **meta}
    restrictions = np.array(SHARED)
    for name, rows in evals.items():
        if preloaded is None:
            logits = get_scores(bank, rows, predictor)
        else:
            positions, raw = preloaded[name]
            logits = np.full((len(rows), len(LABELS)), np.nan)
            if len(positions):
                logits[positions] = predictor(raw)
        unrestricted = predictions_of(logits, np.arange(len(LABELS)))
        restricted = predictions_of(logits, restrictions)
        true = y_of(rows)
        shared_mask = np.isin(true, restrictions)
        present = tuple(sorted(set(true)))
        info = {**score_meta, "eval_ids_sha256": digest_rows(rows), **bank.resources(rows)}
        if len(present) == len(LABELS):
            score_row(info, name, "20way", true, unrestricted, present, LABELS)
        if shared_mask.any():
            refs = tuple(sorted(set(true[shared_mask])))
            score_row(info, name, "shared8_unrestricted20", true[shared_mask], unrestricted[shared_mask], refs, LABELS)
            score_row(info, name, "shared8_restricted8", true[shared_mask], restricted[shared_mask], refs, LABELS)
        pd.DataFrame({"clip_id": [r["clip_id"] for r in rows], "reference": [r["label"] for r in rows],
                      "unrestricted": [LABELS[i] if i >= 0 else "FAILED" for i in unrestricted],
                      "restricted8": [LABELS[i] if i >= 0 else "FAILED" for i in restricted],
                      "scored": unrestricted >= 0}).to_csv(pred_dir / f"{tag}_{name}.csv.gz", index=False)
    if checkpoint is not None:
        folder = OUT / "checkpoints"
        folder.mkdir(exist_ok=True)
        torch.save(checkpoint, folder / f"{tag}.pt")

def checkpoint_payload(model, **kwargs):
    return {"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "labels": LABELS, "layers": LAYERS, **kwargs}

def add_train_resources(meta, rows, bank):
    return {**meta, **{f"train_{k}": v for k, v in bank.resources(rows).items()}}

## 5. EXP-01 · Extreme low-data scaling

All arms receive the same per-layer standardized AraBEST features for each fit. The classical baselines average them uniformly. The neural head learns softmax layer weights and uses the team's 1,000-update budget. A second snapshot uses a training-loss plateau over a fixed subset of at most 2,048 training examples. The optimizer runs until both snapshots are collected or the convergence cap is reached. The fit status distinguishes a plateau from the cap.

In [ ]:
def exp01(sizes):
    if not sizes:
        return
    eval_rows = unique(r for rows in EVAL.values() for r in rows)
    ensure_features(eval_rows, MEAN)
    if sizes != ["full"]:
        largest = max(sizes)
        target = unique(r for seed in SEEDS for r in subset(largest, seed))
        try:
            ensure_features(target, MEAN)
        except ExtractionPaused:
            pass  # Smaller completed subsets can still be scored.
    for size in sizes:
        draw = {seed: subset(size, seed) for seed in SEEDS}
        ensure_features(unique(r for rows in draw.values() for r in rows), MEAN)
        for seed, rows in draw.items():
            print(f"EXP-01 · {len(rows):,} train clips · seed {seed}", flush=True)
            require_train(rows, MEAN)
            save_assignment(f"exp01_train_{size}_{seed}", rows)
            raw = MEAN.load(rows)
            stats = standardizer(raw)
            x = zscore(raw, stats, inplace=True)
            y = y_of(rows)
            baseline_x = pool_mean(x)
            lr, lr_meta = run_lr(baseline_x, y, seed)
            lr_meta = add_train_resources(lr_meta, rows, MEAN)
            report_dialects(1, f"{size}_logreg", seed, rows, EVAL, MEAN,
                            lambda raw: lr_scores(lr, pool_mean(zscore(raw, stats)), len(LABELS)), lr_meta)
            tick = time.perf_counter()
            centers = np.stack([baseline_x[y == i].mean(0) for i in range(len(LABELS))])
            cost = np.square(centers).sum(1)
            centroid_meta = {"trainable_params": 0, "fitted_centroid_values": int(centers.size),
                             "train_seconds": time.perf_counter() - tick, "fit_status": "closed_form", "updates": 0}
            report_dialects(1, f"{size}_centroid", seed, rows, EVAL, MEAN,
                            lambda raw: 2 * pool_mean(zscore(raw, stats)) @ centers.T - cost,
                            add_train_resources(centroid_meta, rows, MEAN))
            del baseline_x
            model, snapshots = mixture(x, y, seed, convergence=True)
            del raw, x
            for protocol, (state, stats_fit) in snapshots.items():
                model.load_state_dict(state)
                arm = f"{size}_mixture_{protocol}"
                report_dialects(1, arm, seed, rows, EVAL, MEAN,
                                lambda raw: torch_scores(model, zscore(raw, stats)),
                                add_train_resources(stats_fit, rows, MEAN),
                                checkpoint=checkpoint_payload(model, mean=torch.from_numpy(stats[0]), std=torch.from_numpy(stats[1]),
                                                              method="LayerMixture", train_ids_sha256=digest_rows(rows)))
            gc.collect()

## 6. EXP-02 · Where dialect information is accessible

Indices 0 and 25 represent the front end and final normalization. Indices 1–24 correspond to the Conformer blocks. The early, middle and late comparisons use blocks 1–8, 9–16 and 17–24. The project's configured middle subset uses blocks 6–18. Both uniform and learned versions of that subset are scored.

In [ ]:
def exp02():
    ensure_features(unique(r for split in EVAL.values() for r in split), MEAN)
    layer_groups = {
        "early_uniform": list(range(1, 9)), "middle_uniform": list(range(9, 17)),
        "late_uniform": list(range(17, 25)), "all_uniform": list(range(len(LAYERS))),
        "project_selected_uniform": SELECTED,
    }
    layer_groups.update({f"layer_{i:02d}": [i] for i in range(len(LAYERS))})
    for seed in SEEDS:
        rows = subset(TRAIN_SIZE, seed)
        ensure_features(rows, MEAN)
        require_train(rows, MEAN)
        save_assignment(f"exp02_train_{seed}", rows)
        x = MEAN.load(rows)
        stats = standardizer(x)
        zscore(x, stats, inplace=True)
        y = y_of(rows)
        # The eval arrays are shared by all 33 probes to avoid reloading cache parts for each layer.
        banks = {name: (positions, zscore(MEAN.load(valid), stats, inplace=True))
                 for name, eval_rows in EVAL.items() for positions, valid in [readable(MEAN, eval_rows)]}
        for arm, layers in layer_groups.items():
            print(f"EXP-02 · {arm} · seed {seed}", flush=True)
            lr, meta = run_lr(pool_mean(x, layers), y, seed)
            cached_predict = lambda raw: lr_scores(lr, pool_mean(raw, layers), len(LABELS))
            report_dialects(2, arm, seed, rows, EVAL, MEAN, cached_predict,
                            add_train_resources(meta, rows, MEAN), preloaded=banks)
        for arm, layers in (("learned_all", list(range(len(LAYERS)))), ("project_selected_learned", SELECTED)):
            feature_input = x if len(layers) == len(LAYERS) else x[:, layers]
            model, snapshots = mixture(feature_input, y, seed)
            state, meta = snapshots["fixed"]
            model.load_state_dict(state)
            weights = model.layer_weights().detach().cpu().numpy()
            pd.DataFrame({"layer": np.asarray(LAYERS)[layers], "weight": weights}).to_csv(
                OUT / f"exp02_{arm}_{seed}_weights.csv", index=False)
            report_dialects(2, arm, seed, rows, EVAL, MEAN, lambda raw: torch_scores(model, raw[:, layers]),
                            add_train_resources(meta, rows, MEAN), preloaded=banks,
                            checkpoint=checkpoint_payload(model, selected_layers=layers,
                                                          mean=torch.from_numpy(stats[0]), std=torch.from_numpy(stats[1]),
                                                          method="LayerMixture"))
        del x, banks

## 7. EXP-03 · Controlled audio perturbations

By default, the attached scaling archive supplies the three saved 16K heads and their original normalization. Set `PERTURB_HEAD_SIZE = 8000` to use the 8K head. If the 16K archive is absent in auto mode, the notebook uses the earlier 8K fusion archive or refits the original 8K recipe. Clean and transformed audio use the same classifier, clips and normalization. Perturbations are pitch ±2 semitones, speed ×0.85/×1.15, equal-length segment shuffling at 250/500 ms, a 300–3400 Hz bandpass, additive noise at 20/10 dB SNR and RMS normalization. A partial segment at the end stays in place. The paired CSV gives clean minus perturbed macro-F1 per seed and split. Changes measure sensitivity to the intervention, and multiple audio properties can change together.

In [ ]:
def read_result_file(source, path):
    if source.is_dir():
        return (source / path).read_bytes()
    with ZipFile(source) as archive:
        return archive.read(path.as_posix())


def verify_scaling_archive():
    protocol = json.loads(read_result_file(SCALING_SOURCE, Path("protocol.json")))
    expected_manifests = {
        **EVAL_HASHES,
        "train_scaling_8000": "c3e90a5441c1d70596c635c0a5e76147c88101b1498c22e76f167090fcea842d",
        "train_scaling_16000": "19063cdde6e28cc96da4d828c08eeb37b62ed8d5f698840fb8340107ddd4b820",
    }
    if (protocol.get("repo_commit") != COMMIT
            or protocol.get("encoder") != CONFIG["encoder"]
            or protocol.get("manifest_sha256") != expected_manifests
            or protocol.get("optimizer_updates_per_head") != 1000
            or protocol.get("seeds") != [490, 491, 492]):
        raise ValueError("The attached 8K/16K results use a different AraBEST recipe, manifest or run protocol.")
    return protocol


def export_scaling_reference():
    if SCALING_SOURCE is None:
        return None
    protocol = verify_scaling_archive()
    raw = read_result_file(SCALING_SOURCE, Path("results.csv"))
    results = pd.read_csv(io.BytesIO(raw))

    expected_sizes = {"ADI20_dev": 200, "Casablanca": 6819}
    expected = {(size, split, seed, n) for size in (8000, 16000)
                for split, n in expected_sizes.items() for seed in protocol["seeds"]}
    actual = set(zip(results.train_size, results.split, results.seed, results.n))
    metrics = ["accuracy", "macro_f1", "jor_recall"]
    if (len(results) != len(expected) or actual != expected
            or not results[metrics].apply(lambda column: column.between(0, 1)).all().all()):
        raise ValueError("The 8K/16K results are missing a matched arm, cohort, seed or valid metric.")

    results["split"] = results["split"].map({"ADI20_dev": "ADI20_dev200", "Casablanca": "Casablanca_full6819"})
    results["optimizer_updates"] = protocol["optimizer_updates_per_head"]
    results.to_csv(OUT / "archived_8k_vs_16k_metrics_per_seed.csv", index=False)
    summary = results.groupby(["split", "n", "train_size"])[metrics].agg(["mean", "std"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(OUT / "archived_8k_vs_16k_summary.csv", index=False)

    paired = results.pivot(index=["split", "n", "seed"], columns="train_size", values=metrics)
    deltas = pd.DataFrame({f"{metric}_gain_pp": 100 * (paired[metric][16000] - paired[metric][8000])
                           for metric in metrics}).reset_index()
    deltas.to_csv(OUT / "archived_8k_vs_16k_paired_deltas.csv", index=False)
    print("Archived 8K and 16K scaling results · full Casablanca validation")
    display(summary.round(4))
    return {"results_sha256": hashlib.sha256(raw).hexdigest(), "run_protocol": protocol}

In [ ]:
def perturb_head_input():
    """Return the EXP-03 head size, its source, and its directory inside the ZIP."""
    if PERTURB_HEAD_SIZE != 8000 and SCALING_SOURCE:
        return 16000, SCALING_SOURCE, Path("train_16000")
    if PERTURB_HEAD_SIZE == 16000:
        raise FileNotFoundError(f"Add {SCALING_RESULTS_PACKAGE} to use the finished 16K heads.")
    folders = [p for p in INPUT.rglob("train_8000") if p.parent.name == "acoustic_heads" and p.is_dir()]
    source = FUSION_ZIP or next(iter(sorted(folders)), None)
    archive_dir = Path("acoustic_heads/train_8000") if FUSION_ZIP else Path()
    return 8000, source, archive_dir


def exp03_heads():
    size, source, archive_dir = perturb_head_input()
    if source is not None and not COHORT_ASSIGNMENTS_FILE:
        raise ValueError("Saved historical EXP-03 heads require the matching local cohort assignments. "
                         "For the public cohorts, omit the saved-head inputs to refit the 8K head.")
    rows = ANCHORS[size]
    trained = {}
    if source is None:
        ensure_features(rows, MEAN)
        require_train(rows, MEAN)
        raw = MEAN.load(rows)
        stats = (raw.mean(0), np.maximum(raw.std(0), CONFIG["preprocessing"]["std_floor"]))
        zscore(raw, stats, inplace=True)
        for seed in SEEDS:
            model, snapshots = mixture(raw, y_of(rows), seed, steps=1000)
            _, meta = snapshots["fixed"]
            trained[seed] = (model.cpu(), {**meta, "head_origin": "refit_8k_recipe"})
        del raw
        for seed, (model, _) in trained.items():
            folder = OUT / "checkpoints"
            folder.mkdir(exist_ok=True)
            torch.save(checkpoint_payload(model, method="LayerMixture", mean=torch.from_numpy(stats[0]),
                                          std=torch.from_numpy(stats[1])), folder / f"exp03_refit_8k_{seed}.pt")
        return rows, stats, trained

    if size == 16000:
        verify_scaling_archive()

    def read(name):
        return read_result_file(source, archive_dir / name)

    with np.load(io.BytesIO(read("zscore_stats.npz")), allow_pickle=False) as values:
        assert values["layer_names"].tolist() == LAYERS
        stats = values["mean"], values["std"]
    assert stats[0].shape == stats[1].shape == (len(LAYERS), WIDTH)
    assert np.isfinite(stats[0]).all() and np.isfinite(stats[1]).all() and (stats[1] > 0).all()
    for seed in SEEDS:
        saved = torch.load(io.BytesIO(read(f"head_{seed}.pt")), map_location="cpu", weights_only=True)
        assert saved["labels"] == LABELS and saved["layers"] == LAYERS
        assert saved["seed"] == seed and saved["train_size"] == len(rows) and saved["updates"] == 1000
        assert clf.feature_config(saved["config"]["encoder"]) == FEATURE_CONFIG
        model = lm.build_model((1, len(LAYERS), WIDTH), len(LABELS), 0)
        model.load_state_dict(saved["state_dict"])
        trained[seed] = (model.eval(), {"train_seconds": np.nan, "trainable_params": count_params(model),
                                       "updates": 1000, "head_origin": f"saved_historical_{size // 1000}k_head"})
    return rows, stats, trained

def exp03():
    rows, stats, trained = exp03_heads()
    all_eval = unique(r for values in EVAL.values() for r in values)
    ensure_features(all_eval, MEAN)
    paused = False
    try:
        ensure_features(all_eval, *AUGMENTED.values())
    except ExtractionPaused:
        paused = True
    for name, bank in {"clean": MEAN, **AUGMENTED}.items():
        if bank.missing(all_eval):
            continue
        for seed, (model, meta) in trained.items():
            print(f"EXP-03 · {len(rows) // 1000}K head · {name} · seed {seed}", flush=True)
            model.to(DEVICE)
            report_dialects(3, name, seed, rows, EVAL, bank, lambda raw: torch_scores(model, zscore(raw, stats)),
                            add_train_resources(meta, rows, MEAN))
            model.cpu()
    if paused:
        raise ExtractionPaused

## 8. EXP-04 · Domain information leakage

Balance each corpus × dialect cell in the fitting and held-out partitions. Keep known speakers, inferred recording or episode groups, and exact duplicates together. ADI-20 metadata leaves many groups unknown. Dialect and corpus accuracy use the same holdouts and representations. A low corpus score indicates limited recoverability by this linear probe.

In [ ]:
def exp04():
    def binary_target(rows):
        return np.array([int(cas(r)) for r in rows])

    for seed, (fit, test) in PROBE_SPLITS.items():
        ensure_features(fit + test, MEAN)
        require_train(fit + test, MEAN)
        raw = MEAN.load(fit)
        stats = standardizer(raw)
        x = zscore(raw, stats, inplace=True)
        xt = zscore(MEAN.load(test), stats, inplace=True)
        for layer, layer_name in enumerate(LAYERS):
            print(f"EXP-04 · {layer_name} · seed {seed}", flush=True)
            fit_x, test_x = x[:, layer], xt[:, layer]
            for task, true_train, true_test, target_names, refs in (
                ("domain", binary_target(fit), binary_target(test), ["ADI20", "Casablanca"], (0, 1)),
                ("dialect", y_of(fit), y_of(test), LABELS, SHARED),
            ):
                lr, meta = run_lr(fit_x, true_train, seed)
                scores = lr_scores(lr, test_x, len(target_names))
                pred = predictions_of(scores, np.asarray(refs))
                info = {**add_train_resources(meta, fit, MEAN), **MEAN.resources(test),
                        "experiment": 4, "seed": seed, "arm": f"layer_{layer:02d}",
                        "train_n": len(fit), "train_ids_sha256": digest_rows(fit), "eval_ids_sha256": digest_rows(test),
                        "group_metadata_coverage": np.mean([r["clip_id"] in KNOWN_GROUPS for r in fit + test])}
                score_row(info, "grouped_balanced_holdout", task, true_test, pred, refs, target_names)
                folder = OUT / "predictions"
                folder.mkdir(exist_ok=True)
                pd.DataFrame({"clip_id": [r["clip_id"] for r in test], "reference": [target_names[i] for i in true_test],
                              "prediction": [target_names[i] if i >= 0 else "FAILED" for i in pred]}).to_csv(
                    folder / f"exp04_{layer:02d}_{task}_{seed}.csv.gz", index=False)

## 9. EXP-05 · Supervised contrastive learning

The projection and 20-way head share normalized embeddings. Balanced batches draw multiple recordings per dialect when possible. The contrastive term prefers positives from different recording groups and falls back to other clips of the same dialect.

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, width, normalize=False, domains=0):
        super().__init__()
        self.project = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))
        self.classifier = nn.Linear(128, len(LABELS))
        self.normalize = normalize
        self.domain = (nn.Sequential(nn.Linear(128 + len(LABELS), 64), nn.GELU(), nn.Linear(64, domains))
                       if domains else None)

    def embedding(self, x):
        z = self.project(x)
        return F.normalize(z, dim=1) if self.normalize else z

    def forward(self, x):
        return self.classifier(self.embedding(x))

class ReverseGradient(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x.view_as(x)

    @staticmethod
    def backward(ctx, gradient):
        return -gradient

def shuffled_dialect_batches(y, groups, seed, per_class=4):
    rng = np.random.default_rng(seed)
    indices_by_label = defaultdict(lambda: defaultdict(list))
    for i, (label, group) in enumerate(zip(y, groups)):
        indices_by_label[label][group].append(i)
    classes = list(indices_by_label.values())
    while True:
        batch = []
        for by_group in classes:
            selected_groups = rng.permutation(list(by_group))[:per_class]
            picks = [int(rng.choice(by_group[g])) for g in selected_groups]
            if len(picks) < per_class:
                remaining = list(set(i for ids in by_group.values() for i in ids) - set(picks))
                picks.extend(rng.choice(remaining, size=min(per_class - len(picks), len(remaining)), replace=False))
            batch.extend(picks)
        yield rng.permutation(batch)

def supcon(z, y, groups, temperature=.1):
    logits = z @ z.T / temperature
    diag = torch.eye(len(y), dtype=torch.bool, device=DEVICE)
    same_label = (y[:, None] == y[None, :]) & ~diag
    preferred = same_label & (groups[:, None] != groups[None, :])
    positive = torch.where(preferred.any(1, keepdim=True), preferred, same_label)
    valid = positive.any(1)
    log_prob = logits - torch.logsumexp(logits.masked_fill(diag, -torch.inf), dim=1, keepdim=True)
    per_anchor = -log_prob.masked_fill(~positive, 0).sum(1) / positive.sum(1).clamp_min(1)
    return per_anchor.masked_fill(~valid, 0).sum() / valid.sum().clamp_min(1)

def projection_input(rows):
    require_train(rows, MEAN)
    raw = MEAN.load(rows)
    stats = standardizer(raw)
    zscore(raw, stats, inplace=True)
    return pool_mean(raw, SELECTED), stats

def projected_results(exp, arm, seed, model, rows, evals, stats, meta):
    report_dialects(exp, arm, seed, rows, evals, MEAN,
                    lambda raw: torch_scores(model, pool_mean(zscore(raw, stats), SELECTED)),
                    add_train_resources(meta, rows, MEAN),
                    checkpoint=checkpoint_payload(model, method="ProjectionHead", arm=arm,
                                                  selected_layers=SELECTED, unit_normalized=model.normalize,
                                                  num_domains=0 if model.domain is None else model.domain[-1].out_features,
                                                  mean=torch.from_numpy(stats[0]), std=torch.from_numpy(stats[1])))

def exp05():
    ensure_features(unique(r for rows in EVAL.values() for r in rows), MEAN)
    for seed in SEEDS:
        rows = subset(TRAIN_SIZE, seed)
        ensure_features(rows, MEAN)
        x, stats = projection_input(rows)
        y = y_of(rows)
        groups = np.unique([GROUPS[r["clip_id"]] for r in rows], return_inverse=True)[1]
        save_assignment(f"exp05_train_{seed}", rows)
        for weight in (0., .1, .5, 1.):
            print(f"EXP-05 · lambda {weight:g} · seed {seed}", flush=True)
            torch.manual_seed(seed)
            model = ProjectionHead(WIDTH, normalize=True).to(DEVICE)
            batches = shuffled_dialect_batches(y, groups, seed)

            def objective(net, inputs, target, indices):
                z = net.embedding(*inputs)
                ce = F.cross_entropy(net.classifier(z), target)
                if weight == 0:
                    return ce
                return ce + weight * supcon(z, target, torch.as_tensor(groups[indices], device=DEVICE))

            result = fit_torch(model, x, y, steps=UPDATES, seed=seed,
                               optimizer=torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.01),
                               batches=batches, objective=objective)
            _, meta = result["fixed"]
            projected_results(5, f"supcon_{weight:g}", seed, model, rows, EVAL, stats,
                              {**meta, "loss_weight": weight})

## 10. EXP-06 · Dialect-conditional domain suppression

Gradient reversal acts on the projected embedding and the adversary also receives the reference dialect. The default is **supervised adaptation**, with 800 labelled Casablanca fitting clips and a 600-clip target holdout from disjoint recording groups. Compare every adversarial weight with a λ = 0 control trained on the same labelled clips. A separate linear probe measures corpus recoverability on a balanced held-out cohort.

To use training-source domains instead, set `SOURCE_DOMAINS_CSV` to a mounted CSV with verified `clip_id,domain` pairs. At least two domains must cover each of the eight shared dialects in the selected training subset. In that setting Casablanca-800 is used only for evaluation.

In [ ]:
def exp06_domains(rows, seed):
    buckets = defaultdict(lambda: defaultdict(list))
    for i, row in enumerate(rows):
        domain = ("Casablanca" if cas(row) else "ADI20") if ADAPT_TRAIN else SOURCE_DOMAINS.get(row["clip_id"])
        label = LABEL_ID[row["label"]]
        if domain and label in SHARED:
            buckets[domain][label].append(i)
    eligible = {d: by_label for d, by_label in buckets.items() if set(by_label) == set(SHARED)}
    if len(eligible) < 2:
        raise ValueError("EXP-06 needs at least two source domains with every shared dialect in TRAIN_SIZE. "
                         "Increase TRAIN_SIZE, revise verified metadata, or clear SOURCE_DOMAINS_CSV for adaptation.")
    names = sorted(eligible)
    quota = min(100, *(len(v) for by_label in eligible.values() for v in by_label.values()))
    rng = np.random.default_rng(seed)
    indices, domains = [], []
    for domain_id, name in enumerate(names):
        for label in SHARED:
            indices.extend(rng.choice(eligible[name][label], size=quota, replace=False))
            domains.extend([domain_id] * quota)
    return np.asarray(indices), np.asarray(domains), names

def adaptation_domain_probe(model, seed, arm, x, stats, domain_y, fit_rows, eval_rows):
    def embedding(features):
        with torch.inference_mode():
            return np.concatenate([model.embedding(torch.as_tensor(block, device=DEVICE)).cpu().numpy()
                                   for block in np.array_split(features, max(1, math.ceil(len(features) / 128)))])
    _, valid = readable(MEAN, eval_rows)
    if len(valid) != len(eval_rows):
        raise ValueError("Missing features in the balanced adaptation domain holdout.")
    raw = zscore(MEAN.load(valid), stats, inplace=True)
    test_x = embedding(pool_mean(raw, SELECTED))
    probe, probe_meta = run_lr(embedding(x), domain_y, seed)
    domain_names = ["ADI20", "Casablanca"]
    truth = np.array([int(cas(row)) for row in valid])
    pred = predictions_of(lr_scores(probe, test_x, 2), np.arange(2))
    meta = {**add_train_resources(probe_meta, fit_rows, MEAN), **MEAN.resources(valid),
            "experiment": 6, "arm": arm, "seed": seed, "train_n": len(fit_rows),
            "train_ids_sha256": digest_rows(fit_rows), "eval_ids_sha256": digest_rows(valid)}
    score_row(meta, "balanced_adaptation_domain_holdout", "domain", truth, pred, (0, 1), domain_names)

def exp06():
    evals = dict(EVAL)
    domain_eval = []
    if ADAPT_TRAIN:
        evals.pop("Casablanca_fixed800")
        evals["Casablanca_adaptation_holdout"] = ADAPT_TEST
        domain_eval = stratified_sample(DEV8, SHARED, len(ADAPT_TEST) // len(SHARED), FIXED_DRAW) + ADAPT_TEST
        save_assignment("exp06_balanced_domain_holdout", domain_eval)
    ensure_features(domain_eval + [r for rows in evals.values() for r in rows], MEAN)
    for seed in SEEDS:
        rows = subset(TRAIN_SIZE, seed) + ADAPT_TRAIN
        ensure_features(rows, MEAN)
        x, stats = projection_input(rows)
        y = y_of(rows)
        domain_rows, domain_y, domain_names = exp06_domains(rows, seed)
        pd.DataFrame({"clip_id": [rows[i]["clip_id"] for i in domain_rows],
                      "dialect": [LABELS[y[i]] for i in domain_rows],
                      "domain": [domain_names[i] for i in domain_y]}).to_csv(
            OUT / f"exp06_{seed}_domain_train.csv.gz", index=False)
        save_assignment(f"exp06_train_{seed}", rows)
        protocol = "supervised_Casablanca_adaptation" if ADAPT_TRAIN else "ADI_source_domain_generalization"
        for weight in (0., .1, .5, 1.):
            print(f"EXP-06 · {protocol} · lambda {weight:g} · seed {seed}", flush=True)
            torch.manual_seed(seed)
            model = ProjectionHead(WIDTH, domains=len(domain_names)).to(DEVICE)
            if weight == 0:
                model.domain.requires_grad_(False)
            batches = mini_batches(len(domain_y), 128, seed + 101)

            def objective(net, inputs, target, indices):
                loss = F.cross_entropy(net(*inputs), target)
                if weight == 0:
                    return loss
                j = next(batches)
                original = domain_rows[j]
                hidden = net.embedding(*collate(x, original))
                dialect = F.one_hot(torch.as_tensor(y[original], device=DEVICE), num_classes=len(LABELS))
                conditioned = torch.cat([ReverseGradient.apply(hidden), dialect.to(hidden.dtype)], dim=1)
                domain_loss = F.cross_entropy(net.domain(conditioned), torch.as_tensor(domain_y[j], device=DEVICE))
                return loss + weight * domain_loss

            result = fit_torch(model, x, y, steps=UPDATES, seed=seed,
                               optimizer=torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.01),
                               objective=objective)
            _, meta = result["fixed"]
            arm = f"adversarial_{weight:g}"
            projected_results(6, arm, seed, model, rows, evals, stats,
                              {**meta, "loss_weight": weight, "protocol": protocol,
                               "domain_train_n": len(domain_y), "training_domains": len(domain_names)})
            if ADAPT_TRAIN:
                domain_fit = [rows[i] for i in domain_rows]
                check_disjoint(domain_fit, domain_eval, GROUPS)
                adaptation_domain_probe(model, seed, arm, x[domain_rows], stats, domain_y,
                                        domain_fit, domain_eval)

## 11. EXP-07 · Frame pooling

Reuse the same selected-layer frame features across all four heads. Padding receives zero weight. The statistics arms concatenate the population mean and standard deviation, and the attention arms learn one weight per valid frame.

In [ ]:
class PoolingHead(nn.Module):
    def __init__(self, width, mode):
        super().__init__()
        self.mode = mode
        self.attention = nn.Linear(width, 1) if mode.startswith("attentive") else None
        pooled_width = width * (2 if "statistics" in mode else 1)
        self.head = nn.Sequential(nn.LayerNorm(pooled_width), nn.Linear(pooled_width, 256),
                                  nn.GELU(), nn.Linear(256, len(LABELS)))

    def forward(self, frames, mask):
        if self.attention is None:
            alpha = mask / mask.sum(1, keepdim=True)
        else:
            alpha = self.attention(frames).squeeze(-1).masked_fill(~mask, -torch.inf).softmax(1)
        weights = alpha.unsqueeze(-1)
        mean = (frames * weights).sum(1)
        if "statistics" in self.mode:
            variance = (weights * (frames - mean.unsqueeze(1)).square()).sum(1)
            mean = torch.cat([mean, variance.clamp_min(1e-6).sqrt()], dim=1)
        return self.head(mean)

def exp07():
    ensure_features(unique(r for rows in EVAL.values() for r in rows), FRAMES)
    for seed in SEEDS:
        rows = subset(POOL_SIZE, seed)
        ensure_features(rows, FRAMES)
        require_train(rows, FRAMES)
        x = FRAMES.load(rows)
        y = y_of(rows)
        save_assignment(f"exp07_train_{seed}", rows)
        banks = {name: (positions, FRAMES.load(valid))
                 for name, eval_rows in EVAL.items() for positions, valid in [readable(FRAMES, eval_rows)]}
        for arm in ("mean", "statistics", "attentive", "attentive_statistics"):
            print(f"EXP-07 · {arm} · seed {seed}", flush=True)
            torch.manual_seed(seed)
            model = PoolingHead(WIDTH, arm).to(DEVICE)
            result = fit_torch(model, x, y, steps=UPDATES, seed=seed, batch_size=32,
                               optimizer=torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.01))
            _, meta = result["fixed"]
            report_dialects(7, arm, seed, rows, EVAL, FRAMES, lambda raw: torch_scores(model, raw),
                            add_train_resources(meta, rows, FRAMES), preloaded=banks,
                            checkpoint=checkpoint_payload(model, method="PoolingHead", mode=arm,
                                                          selected_layers=SELECTED))

## 12. E11 · Variational information bottleneck

**Question.** At matched supervision and optimizer budget, does a compressed latent improve dialect recognition on full Casablanca validation? Can a source-fitted domain-distance score recover the corpus less accurately after compression?

**Protocol.** Freeze AraBEST and reuse the same verified pooled per-layer embeddings. Fit normalization on each ADI-20 training subset and uniformly average the same selected layers used by the project's projection MLP. Use three seeds across all five dimensions and four values of `β` at 4K. At 4K, also fit the project's cross-entropy projection MLP and deterministic bottlenecks at the five matching dimensions. The deterministic 128-dimensional control has the same architecture as the existing MLP baseline. The archived 16K layer-mixture results remain available as a separate reference.

Choose one positive-β configuration by mean ADI-20 dev macro-F1 across seeds before running Casablanca inference. Break ties by mean accuracy, lower dimension and lower β. At 1K and 16K, refit that configuration, the MLP, and same-width deterministic and β=0 controls. The ADI-20 results are development scores because the 200-clip manifest guides selection. Casablanca has the same 6,819 validation rows as the earlier scaling run and contributes only to evaluation. Report 20-way predictions on both corpora, computing Casablanca macro-F1 over its eight reference dialects. Inference uses the posterior mean for every variational arm.

**Domain diagnostic.** Before and after compression, fit a diagonal distance score to each dialect's ADI-20 training embeddings. Score the shared-dialect ADI-20 dev cohort and Casablanca validation with those same source-fitted statistics. Report within-dialect corpus AUROC and its distance from chance. This measures domain separation detectable by that score. Speaker identity and invariance require other evidence.

### Deriving the objective

Let \(X\) be the frozen AraBEST vector, \(Y\) the dialect and \(Z\) the latent. For a positive β, minimize

\[
\beta I(X;Z)-I(Y;Z).
\]

Average over the data distribution and the variational encoder \(q_\theta(z\mid x)\). If \(p_\phi(y\mid z)\) is the trained classifier, its cross-entropy equals

\[
\mathbb E[-\log p_\phi(Y\mid Z)]
  = H(Y\mid Z) + \mathbb E_Z D_{\mathrm{KL}}(p(Y\mid Z)\Vert p_\phi(Y\mid Z)).
\]

It bounds \(H(Y\mid Z)\) from above. Since \(I(Y;Z)=H(Y)-H(Y\mid Z)\) and \(H(Y)\) is fixed, this supplies the prediction term. For any prior \(r(z)\),

\[
\mathbb E_X D_{\mathrm{KL}}(q_\theta(Z\mid X)\Vert r(Z))
   = I(X;Z)+D_{\mathrm{KL}}(q_\theta(Z)\Vert r(Z)),
\]

where \(q_\theta(z)=\mathbb E_X q_\theta(z\mid X)\). Add and subtract \(\log q_\theta(z)\) in the expected log ratio to obtain the identity. Thus the expected prior KL upper-bounds \(I(X;Z)\).

Use \(r=\mathcal N(0,I)\) and \(q_\theta(z\mid x)=\mathcal N(\mu_\theta(x), \operatorname{diag}(e^{s_\theta(x)}))\), where \(s\) is log-variance. Sample \(z=\mu+e^{s/2}\odot\epsilon\), with \(\epsilon\sim\mathcal N(0,I)\). The diagonal Gaussian identity gives

\[
D_{\mathrm{KL}}(q_\theta(z\mid x)\Vert r(z))
= \tfrac12\sum_{j=1}^{d}(\mu_j^2+e^{s_j}-1-s_j).
\]

Optimize the batch-mean cross-entropy plus β times the batch-mean KL. The β=0 control retains the stochastic latent and drops the KL penalty.

See [Tishby, Pereira and Bialek](https://arxiv.org/abs/physics/0004057), [Alemi et al.](https://arxiv.org/abs/1612.00410), and [Tabbara, Fahs and Assaad](https://arxiv.org/abs/2610.25952). The Markov-chain analysis provides related motivation. The implemented objective follows the variational formulation.

**Outputs.** The results ZIP contains the E11 heads, the 4K sweep and selection record, metric CSVs and JSON, comparison plots, and a short analysis generated from the completed runs. Compatible fitted heads can be reused across resumed sessions by attaching previous outputs.

In [ ]:
from sklearn.metrics import roc_auc_score

E11_DIMS = (16, 32, 64, 128, 256)
E11_BETAS = (0., .001, .01, .1)
E11_MLP = ("mlp", 128, 0.)
E11_COHORTS = {"ADI20_dev200": DEV20, "Casablanca_full6819": CAS_ALL}
E11_DOMAIN = []

class VariationalDialectHead(nn.Module):
    def __init__(self, latent_dim, variational):
        super().__init__()
        self.project = nn.Sequential(nn.Linear(WIDTH, 256), nn.GELU(), nn.Linear(256, latent_dim))
        self.classifier = nn.Linear(latent_dim, len(LABELS))
        self.logvar = nn.Linear(256, latent_dim) if variational else None

    def posterior(self, x):
        hidden = self.project[1](self.project[0](x))
        mean = self.project[2](hidden)
        logvar = self.logvar(hidden).clamp(-12., 8.) if self.logvar is not None else None
        return mean, logvar

    def embedding(self, x):
        return self.posterior(x)[0]

    def forward(self, x):
        return self.classifier(self.embedding(x))

def diagonal_kl(mean, logvar):
    return .5 * (mean.square() + logvar.exp() - logvar - 1).sum(dim=-1)

def e11_model(arm):
    kind, dim, _ = arm
    if kind == "mlp":
        return ProjectionHead(WIDTH)
    return VariationalDialectHead(dim, variational=kind == "vib")

def e11_name(arm):
    kind, dim, beta = arm
    return "mlp" if kind == "mlp" else f"{kind}_d{dim}_b{beta:g}"

def e11_features(rows, stats=None):
    raw = MEAN.load(rows)[:, SELECTED]
    stats = standardizer(raw) if stats is None else stats
    return pool_mean(zscore(raw, stats, inplace=True)), stats

def e11_require_features(rows, set_name):
    missing = MEAN.missing(unique(rows))
    if not missing:
        return
    if not E11_ENCODE_MISSING:
        raise ExtractionPaused(f"E11 {set_name}: {len(missing):,} pooled rows missing. Attach the AraBEST cache "
                               "or set E11_ENCODE_MISSING = True to encode only those rows.")
    ensure_features(missing, MEAN)

def e11_fit_head(arm, x, y, seed):
    torch.manual_seed(seed)
    model = e11_model(arm).to(DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(seed + 111)
    kind, _, beta = arm

    def loss(model, inputs, target, _):
        if kind != "vib":
            return F.cross_entropy(model(*inputs), target)
        mean, logvar = model.posterior(*inputs)
        epsilon = torch.randn(mean.shape, device=mean.device, dtype=mean.dtype, generator=generator)
        z = mean + logvar.mul(.5).exp() * epsilon
        return F.cross_entropy(model.classifier(z), target) + beta * diagonal_kl(mean, logvar).mean()

    optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.01)
    result = fit_torch(model, x, y, steps=UPDATES, seed=seed, optimizer=optimizer, objective=loss)
    return model, result["fixed"][1]

@torch.inference_mode()
def e11_inference(model, features):
    model.eval()
    means, logits, kls = [], [], []
    for block in np.array_split(features, max(1, math.ceil(len(features) / 512))):
        x = torch.as_tensor(block, device=DEVICE)
        if isinstance(model, VariationalDialectHead):
            mean, logvar = model.posterior(x)
            if logvar is not None:
                kls.append(diagonal_kl(mean, logvar).cpu().numpy())
        else:
            mean = model.embedding(x)
        means.append(mean.cpu().numpy())
        logits.append(model.classifier(mean).cpu().numpy())
    return np.concatenate(means), np.concatenate(logits), float(np.concatenate(kls).mean()) if kls else None

def e11_fit_size(size, arms, output, previous_heads):
    records = []
    train_by_seed = {seed: subset(size, seed) for seed in SEEDS}
    e11_require_features([r for rows in train_by_seed.values() for r in rows], f"{size:,} training")
    _, valid_dev = readable(MEAN, DEV20)
    for seed, rows in train_by_seed.items():
        if any(r["source_dataset"] != ADI_SOURCE or r["split"] != "train" for r in rows):
            raise ValueError("E11 fitting is restricted to ADI-20 train.")
        require_train(rows, MEAN)
        save_assignment(f"e11_train_{size}_{seed}", rows)
        x, stats = e11_features(rows)
        if A100_40GB:
            x = torch.as_tensor(x, device=DEVICE)
        dev, _ = e11_features(valid_dev, stats)
        y = y_of(rows)
        for arm in arms:
            spec = {"recipe": "e11-v1", "arm": list(arm), "seed": seed, "updates": UPDATES,
                    "train_ids_sha256": digest_rows(rows), "selected_layers": list(SELECTED),
                    "feature_config_sha256": PROVENANCE["feature_config_sha256"]}
            key = hashlib.sha256(ae.canonical_json(spec)).hexdigest()[:16]
            path = output / "heads" / f"e11_{size}_{e11_name(arm)}_{seed}_{key}.pt"
            match = previous_heads.get(path.name)
            if match:
                saved = torch.load(match, map_location="cpu", weights_only=True)
                if saved["fit_spec"] != spec or saved["labels"] != LABELS:
                    raise ValueError(f"E11 checkpoint provenance mismatch: {match.name}")
                model = e11_model(arm).to(DEVICE)
                model.load_state_dict(saved["state_dict"])
                if path != match:
                    temp = path.with_suffix(".tmp")
                    shutil.copyfile(match, temp)
                    temp.replace(path)
                meta = saved["metrics"]
            else:
                print(f"E11 · {size:,} · {e11_name(arm)} · seed {seed}", flush=True)
                model, meta = e11_fit_head(arm, x, y, seed)
                payload = checkpoint_payload(model, fit_spec=spec, metrics=meta,
                                             mean=torch.from_numpy(stats[0]), std=torch.from_numpy(stats[1]))
                temp = path.with_suffix(".tmp")
                torch.save(payload, temp)
                temp.replace(path)
                previous_heads[path.name] = path
            _, scores, kl = e11_inference(model, dev)
            positions, _ = readable(MEAN, DEV20)
            predicted = np.full(len(DEV20), -1, dtype=int)
            predicted[positions] = np.argmax(scores, axis=1)
            truth = y_of(DEV20)
            records.append({"train_n": size, "arm": e11_name(arm), "kind": arm[0],
                            "latent_dim": arm[1], "beta": arm[2], "seed": seed, "head": path.name,
                            "dev_accuracy": float((predicted == truth).mean()),
                            "dev_macro_f1": f1_score(truth, predicted, labels=np.arange(len(LABELS)),
                                                     average="macro", zero_division=0),
                            "dev_kl_nats": kl, **meta})
    return pd.DataFrame(records)

def e11_select(sweep):
    score = sweep[(sweep.kind == "vib") & (sweep.beta > 0)].groupby(["latent_dim", "beta"], as_index=False).agg(
        dev_macro_f1=("dev_macro_f1", "mean"), dev_accuracy=("dev_accuracy", "mean"), seeds=("seed", "nunique"))
    if score.empty or not (score.seeds == len(SEEDS)).all():
        raise ValueError("Complete the three-seed 4K VIB sweep before selecting an arm.")
    return score.sort_values(["dev_macro_f1", "dev_accuracy", "latent_dim", "beta"],
                             ascending=[False, False, True, True]).iloc[0].to_dict()

def e11_domain_auc(train_x, train_y, adi_x, adi_y, cas_x, cas_y):
    """One-class ADI-20 distance diagnostic. Casablanca never fits the scores."""
    rows = []
    for dialect in SHARED:
        train = train_x[train_y == dialect]
        adi = adi_x[adi_y == dialect]
        target = cas_x[cas_y == dialect]
        if not len(train) or not len(adi) or not len(target):
            raise ValueError(f"Missing E11 domain diagnostic support for {LABELS[dialect]}.")
        center = train.mean(axis=0)
        scale = train.std(axis=0).clip(1e-4)
        def distance(x):
            return np.square((x - center) / scale).mean(axis=1)
        truth = np.r_[np.zeros(len(adi)), np.ones(len(target))]
        auc = roc_auc_score(truth, np.r_[distance(adi), distance(target)])
        rows.append({"dialect": LABELS[dialect], "adi_eval_n": len(adi),
                     "casablanca_eval_n": len(target), "domain_auroc": auc})
    return rows

def e11_evaluate(fits, output):
    for seed, jobs in fits.groupby("seed", sort=False):
        rows = subset(int(jobs.train_n.iloc[0]), int(seed))
        payload = torch.load(output / "heads" / jobs["head"].iloc[0], map_location="cpu", weights_only=True)
        stats = payload["mean"].numpy(), payload["std"].numpy()
        train_x, _ = e11_features(rows, stats)
        train_y = y_of(rows)
        eval_x, eval_y = {}, {}
        for name, cohort in {**E11_COHORTS, "ADI20_shared800": DEV8}.items():
            positions, valid = readable(MEAN, cohort)
            eval_x[name] = positions, e11_features(valid, stats)[0]
            eval_y[name] = y_of(valid)
        adi = "ADI20_shared800"
        target = "Casablanca_full6819"
        domain_before = e11_domain_auc(train_x, train_y, eval_x[adi][1], eval_y[adi],
                                      eval_x[target][1], eval_y[target])

        for job in jobs.itertuples():
            arm = job.kind, int(job.latent_dim), float(job.beta)
            state = torch.load(output / "heads" / job.head, map_location="cpu", weights_only=True)
            model = e11_model(arm).to(DEVICE)
            model.load_state_dict(state["state_dict"])
            train_z, _, _ = e11_inference(model, train_x)
            inferred = {name: e11_inference(model, x) for name, (_, x) in eval_x.items()}
            domain_after = e11_domain_auc(train_z, train_y, inferred[adi][0], eval_y[adi],
                                          inferred[target][0], eval_y[target])
            tag = f"{job.train_n}_{job.arm}"
            metadata = {**state["metrics"], "kind": job.kind, "latent_dim": job.latent_dim,
                        "beta": job.beta, "head": job.head, "dev_kl_nats": inferred["ADI20_dev200"][2],
                        "casablanca_kl_nats": inferred[target][2]}
            report_dialects(11, tag, int(seed), rows, E11_COHORTS, MEAN, lambda logits: logits,
                            add_train_resources(metadata, rows, MEAN),
                            preloaded={name: (eval_x[name][0], inferred[name][1]) for name in E11_COHORTS})
            for before, after in zip(domain_before, domain_after, strict=True):
                pre, post = before["domain_auroc"], after["domain_auroc"]
                E11_DOMAIN.append({**after, "arm": job.arm, "seed": seed, "train_n": job.train_n,
                                   "before_domain_auroc": pre,
                                   "before_separability": .5 + abs(pre - .5),
                                   "after_separability": .5 + abs(post - .5)})

def e11_tables(output, selected):
    rows = pd.DataFrame([r for r in REPORTS if r["experiment"] == 11
                         and (r["mode"] == "20way" or (r["split"] == "Casablanca_full6819"
                                                      and r["mode"] == "shared8_unrestricted20"))])
    if rows.empty:
        return
    rows["kl_nats"] = np.where(rows.split == "ADI20_dev200", rows["dev_kl_nats"], rows["casablanca_kl_nats"])
    rows.to_csv(output / "metrics_per_seed.csv", index=False)
    fields = ["accuracy", "macro_f1", "kl_nats", "train_seconds", "trainable_params", "coverage"]
    keys = ["train_n", "split", "arm", "kind", "latent_dim", "beta"]
    summary = rows.groupby(keys)[fields].agg(["mean", "std"])
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(output / "metrics_summary.csv", index=False)
    summary.to_json(output / "metrics_summary.json", orient="records", indent=2)

    domain = pd.DataFrame(E11_DOMAIN)
    domain.to_csv(output / "domain_by_seed_and_dialect.csv", index=False)
    columns = ["before_domain_auroc", "domain_auroc", "before_separability", "after_separability"]
    scores = domain.groupby(["train_n", "arm", "seed"])[columns].mean()
    scores["after_minus_before_auroc"] = scores["domain_auroc"] - scores["before_domain_auroc"]
    scores["separability_change"] = scores["after_separability"] - scores["before_separability"]
    domain_summary = scores.groupby(["train_n", "arm"]).agg(["mean", "std"])
    domain_summary.columns = [f"{metric}_{stat}" for metric, stat in domain_summary.columns]
    domain_summary.reset_index().to_csv(output / "domain_summary.csv", index=False)

    chosen = ("vib", int(selected["latent_dim"]), float(selected["beta"]))
    control = ("vib", chosen[1], 0.)
    lines = ["# E11 results", "", f"ADI-20 development selected {chosen[1]} latent dimensions and β = {chosen[2]:g}.",
             "The table compares three-seed mean macro-F1. Deltas are percentage points.", "",
             "| Training clips | Corpus | VIB macro-F1 | Δ vs MLP | Δ vs deterministic | Δ vs β=0 |",
             "|---:|---|---:|---:|---:|---:|"]
    def metric(group, arm):
        return group.loc[group.arm == f"{group.train_n.iloc[0]}_{e11_name(arm)}", "macro_f1_mean"].item()
    for (n, split), group in summary.groupby(["train_n", "split"]):
        vib = metric(group, chosen)
        differences = [100 * (vib - metric(group, a)) for a in (E11_MLP, ("deterministic", chosen[1], 0.), control)]
        lines.append(f"| {n:,} | {split} | {vib:.4f} | {differences[0]:+.2f} | {differences[1]:+.2f} | {differences[2]:+.2f} |")
    lines += ["", "Positive deltas favor the selected VIB at the corresponding training size.",
              "ADI-20 dev was used for hyperparameter choice. Treat its scores as development results.",
              "", "Within-dialect domain separability before and after the selected VIB"]
    for (n, arm), item in domain_summary.iterrows():
        if arm == e11_name(chosen):
            before = item["before_separability_mean"]
            after = item["after_separability_mean"]
            lines.append(f"- {n:,} clips: {before:.3f} before and {after:.3f} after compression.")
    lines += ["", "The distance-based domain score measures a specific distribution shift. Speaker identity requires a separate audit."]
    atomic_text(output / "analysis.md", "\n".join(lines) + "\n")

from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from threading import Event


@contextmanager
def e11_feature_pipeline(jobs):
    """One cache writer prepares later inputs while the heads train."""
    if not (DUAL_GPU_ACTIVE and E11_ENCODE_MISSING):
        yield lambda name: e11_require_features(*jobs[name])
        return

    stop = Event()
    executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="arabest-encoder")
    futures = {}

    def encode(rows):
        with torch.cuda.device(ENCODER_DEVICE):
            ensure_features(rows, MEAN, stop=stop)

    def wait_for(name):
        if name in futures:
            futures[name].result()
        else:
            e11_require_features(*jobs[name])

    try:
        for name, (rows, _) in jobs.items():
            if MEAN.missing(unique(rows)):
                futures[name] = executor.submit(encode, rows)
        if futures:
            print(f"E11 · {DEVICE} head training / {ENCODER_DEVICE} feature prefetch", flush=True)
        yield wait_for
    finally:
        stop.set()
        executor.shutdown(wait=True, cancel_futures=True)


def exp11():
    output = OUT / "E11"
    (output / "heads").mkdir(parents=True, exist_ok=True)
    previous = {p.name: p for p in sorted(INPUT.rglob("e11_*_*.pt"))}
    for pattern in ("adi20_research_*/E11/heads/e11_*.pt",
                    "adi20_research_*/stages/*/E11/heads/e11_*.pt"):
        previous.update({p.name: p for p in sorted(WORK.glob(pattern))})
    e11_require_features(DEV20, "ADI-20 development")
    arms = [E11_MLP] + [("deterministic", d, 0.) for d in E11_DIMS]
    arms += [("vib", d, beta) for d in E11_DIMS for beta in E11_BETAS]
    train4k = [row for seed in SEEDS for row in subset(4000, seed)]
    e11_require_features(train4k, "4,000 training")
    jobs = {"evaluation": (DEV8 + CAS_ALL, "full Casablanca and shared ADI-20 diagnostics")}
    jobs.update({size: ([row for seed in SEEDS for row in subset(size, seed)], f"{size:,} training")
                 for size in E11_EXTEND_SIZES})
    with e11_feature_pipeline(jobs) as wait_for_features:
        sweep = e11_fit_size(4000, arms, output, previous)
        sweep.to_csv(output / "4k_dev_sweep.csv", index=False)
        selected = e11_select(sweep)
        atomic_text(output / "selection.json", json.dumps({
            "train_n": 4000, "selection_split": "ADI20_dev200", "selection_ids_sha256": digest_rows(DEV20),
            "criterion": "Three-seed mean ADI-20 dev macro-F1 among beta > 0, then accuracy, smaller dimension and beta",
            **selected}, indent=2))
        d, beta = int(selected["latent_dim"]), float(selected["beta"])
        print(f"E11 dev-only selection: latent dim {d}, beta {beta:g}", flush=True)
        wait_for_features("evaluation")
        e11_evaluate(sweep, output)
        e11_tables(output, selected)
        save_e11_milestone(output, 4000)
        extension_arms = [E11_MLP, ("deterministic", d, 0.), ("vib", d, 0.), ("vib", d, beta)]
        for size in E11_EXTEND_SIZES:
            wait_for_features(size)
            fitted = e11_fit_size(size, extension_arms, output, previous)
            e11_evaluate(fitted, output)
            e11_tables(output, selected)
            save_e11_milestone(output, size)

## Results and archive helpers

In [ ]:
import matplotlib.pyplot as plt

def export_figures(summary):
    if summary is None:
        return
    comparisons = [("ADI20_dev200", "20way"), ("Casablanca_fixed800", "shared8_unrestricted20")]
    styles = {"logreg": "Logistic regression", "centroid": "Nearest centroid",
              "mixture_fixed": "Layer mixture · fixed updates", "mixture_converged": "Layer mixture · plateau control"}
    scaling = summary[summary.experiment == 1].copy()
    if len(scaling):
        split_arm = scaling.arm.str.split("_", n=1, expand=True)
        scaling["size"] = split_arm[0].replace({"full": str(len(TRAIN))}).astype(int)
        scaling["method"] = split_arm[1]
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
        for axis, (split, mode) in zip(ax, comparisons):
            plotted = scaling[(scaling.split == split) & (scaling["mode"] == mode)]
            for method, points in plotted.groupby("method"):
                points = points.sort_values("size")
                axis.errorbar(points["size"], points.macro_f1_mean, yerr=points.macro_f1_sd.fillna(0),
                              marker="o", label=styles[method], capsize=2)
            axis.set(xscale="log", xlabel="Training clips", title=split.replace("_", " "))
            axis.grid(alpha=.25)
        ax[0].set_ylabel("Macro-F1")
        handles, labels = ax[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
        fig.tight_layout(rect=(0, .14, 1, 1))
        fig.savefig(OUT / "exp01_scaling.png", dpi=180)
        plt.show()
        plt.close(fig)

    layers = summary[(summary.experiment.isin([2, 4])) & summary.arm.str.match(r"layer_\d{2}$")].copy()
    if len(layers):
        layers["index"] = layers.arm.str.rsplit("_", n=1).str[-1].astype(int)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
        for split, mode in comparisons:
            points = layers[(layers.experiment == 2) & (layers.split == split) & (layers["mode"] == mode)].sort_values("index")
            if len(points):
                ax[0].errorbar(points["index"], points.macro_f1_mean, yerr=points.macro_f1_sd.fillna(0),
                               label=split.replace("_", " "), marker=".", capsize=2)
        for task in ("dialect", "domain"):
            points = layers[(layers.experiment == 4) & (layers["mode"] == task)].sort_values("index")
            if len(points):
                ax[1].errorbar(points["index"], points.accuracy_mean, yerr=points.accuracy_sd.fillna(0),
                               label=task.capitalize(), marker=".", capsize=2)
        for axis, title, metric in [(ax[0], "Dialect transfer", "Macro-F1"), (ax[1], "Grouped leakage probes", "Accuracy")]:
            axis.set(xlabel="AraBEST representation index", ylabel=metric, title=title)
            axis.grid(alpha=.25)
            if axis.has_data():
                axis.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(OUT / "exp02_exp04_layers.png", dpi=180)
        plt.show()
        plt.close(fig)

    delta_path = OUT / "exp03_paired_deltas_summary.csv"
    if delta_path.exists():
        deltas = pd.read_csv(delta_path)
        deltas = deltas[(deltas.split.isin(["ADI20_shared800", "Casablanca_fixed800"])) &
                        (deltas["mode"] == "shared8_unrestricted20")]
        if len(deltas):
            means = deltas.pivot(index="arm", columns="split", values="mean_delta").reindex(PERTURBATIONS)
            sds = deltas.pivot(index="arm", columns="split", values="sd_delta").reindex(PERTURBATIONS).fillna(0)
            fig, ax = plt.subplots(figsize=(8, 5))
            means.mul(100).plot.barh(ax=ax, xerr=sds.mul(100), width=.8)
            ax.axvline(0, color="0.4", linewidth=1)
            ax.set(xlabel="Macro-F1 drop (percentage points)", ylabel="", title="Sensitivity to audio perturbations")
            ax.legend(frameon=False)
            fig.tight_layout()
            fig.savefig(OUT / "exp03_perturbation_drops.png", dpi=180)
            plt.show()
            plt.close(fig)

def export_e11_figures():
    folder = OUT / "E11"
    path = folder / "metrics_summary.csv"
    if not path.is_file():
        return
    data = pd.read_csv(path)
    picked = json.loads((folder / "selection.json").read_text())
    dim, beta = int(picked["latent_dim"]), float(picked["beta"])
    comparison = {
        f"{n}_{e11_name(arm)}": label
        for n in data.train_n.unique()
        for arm, label in [(E11_MLP, "MLP"), (("deterministic", dim, 0.), f"Deterministic · {dim}D"),
                           (("vib", dim, 0.), "VIB · β=0"), (("vib", dim, beta), f"VIB · β={beta:g}")]
    }
    data["comparison"] = data.arm.map(comparison)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.6))
    fig.suptitle("E11   |   Compression and cross-corpus transfer", x=.06, ha="left",
                 fontsize=18, fontweight="semibold", color="#182f47")
    palette = ("#275d9a", "#168375", "#bd6428", "#835da0")
    beta_colors = dict(zip(E11_BETAS, palette))
    arm_colors = dict(zip(list(dict.fromkeys(comparison.values())), palette))
    for row, (split, cohort) in enumerate(data.groupby("split", sort=True)):
        title = {"ADI20_dev200": "ADI-20 dev (200 clips)",
                 "Casablanca_full6819": "Casablanca validation (6,819 clips)"}.get(split, split)
        sweep = cohort[(cohort.train_n == 4000) & (cohort.kind == "vib")]
        for regularization, group in sweep.groupby("beta"):
            dim_sorted = group.sort_values("latent_dim")
            axes[row, 0].errorbar(dim_sorted.latent_dim, dim_sorted.macro_f1_mean,
                                  yerr=dim_sorted.macro_f1_std.fillna(0), marker="o", capsize=2,
                                  color=beta_colors[regularization], label=f"β = {regularization:g}")
            axes[row, 1].scatter(group.kl_nats_mean, group.macro_f1_mean,
                                 color=beta_colors[regularization], label=f"β = {regularization:g}")
        for label, group in cohort.dropna(subset=["comparison"]).groupby("comparison"):
            points = group.sort_values("train_n")
            axes[row, 2].errorbar(points.train_n, points.macro_f1_mean, yerr=points.macro_f1_std.fillna(0),
                                  marker="o", capsize=2, color=arm_colors[label], label=label)
        for col, xlabel in enumerate(("Latent dimensions · 4K sweep", "Mean posterior KL (nats) · 4K sweep",
                                      "Training clips · selected configuration and controls")):
            ax = axes[row, col]
            ax.set(xlabel=xlabel, ylabel="Macro-F1")
            ax.set_title(title, loc="left", color="#182f47", fontweight="semibold")
            ax.spines[["top", "right"]].set_visible(False)
            ax.grid(axis="y", alpha=.23)
        axes[row, 0].set_xscale("log", base=2)
        axes[row, 0].set_xticks(E11_DIMS, labels=E11_DIMS)
        axes[row, 2].set_xscale("log")
        axes[row, 2].set_xticks(sorted(cohort.train_n.unique()), labels=sorted(cohort.train_n.unique()))
    for ax in (axes[0, 0], axes[0, 2]):
        ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(rect=(0, 0, 1, .94))
    fig.savefig(folder / "compression_and_transfer.png", dpi=180)
    plt.show()
    plt.close(fig)


from zipfile import ZIP_DEFLATED


def archive_results(name, files):
    ARCHIVES.mkdir(parents=True, exist_ok=True)
    target = ARCHIVES / f"{name}.zip"
    partial = target.with_suffix(".zip.tmp")
    with ZipFile(partial, "w", compression=ZIP_DEFLATED, compresslevel=3) as archive:
        for path, relative in files:
            archive.write(path, arcname=relative)
    partial.replace(target)
    print(f"Saved {target}", flush=True)
    return target


def save_e11_milestone(output, size):
    export_e11_figures()
    files = [(p, f"E11/{p.name}") for p in sorted(output.iterdir()) if p.is_file()]
    files.extend((p, f"E11/heads/{p.name}") for p in sorted((output / "heads").glob(f"e11_{size}_*.pt")))
    files.extend((p, f"predictions/{p.name}") for p in sorted((OUT / "predictions").glob(f"exp11_{size}_*.csv.gz")))
    files.append((SUITE_OUT / "protocol.json", "protocol.json"))
    archive_results(f"E11_{size}_completed", files)


def release_stage_memory():
    global ENCODER
    ENCODER = None
    gc.collect()
    for gpu in range(torch.cuda.device_count()):
        with torch.cuda.device(gpu):
            torch.cuda.empty_cache()


## 13. Check cache coverage and run the selected experiments

Low-data results, representation probes and the remaining ablations run before the 4K–16K and full-data scaling stages. The full-data stage and waveform perturbations can span multiple Kaggle sessions with a cold cache. Use `run_status.csv` and attach prior output caches when continuing a paused stage.

In [ ]:
SUITE_OUT = OUT
ARCHIVES = SUITE_OUT / "archives"

def coverage(name, rows, bank):
    rows = unique(rows)
    missing = len(bank.missing(rows))
    bad = sum(not usable(row) for row in rows)
    return {"set": name, "features": bank.name, "rows": len(rows),
            "cached": len(rows) - missing - bad, "to_encode": missing, "manifest_failures": bad}

def preflight():
    sets = [coverage(name, rows, MEAN) for name, rows in EVAL.items()]
    if 1 in RUN:
        for size in SIZES:
            rows = unique(r for s in SEEDS for r in subset(size, s))
            sets.append(coverage(f"EXP-01 {size} across seeds", rows, MEAN))
    if set(RUN) & {2, 5, 6}:
        sets.append(coverage(f"EXP-02/05/06 {TRAIN_SIZE}", unique(r for seed in SEEDS for r in subset(TRAIN_SIZE, seed)), MEAN))
    if 3 in RUN:
        perturb_rows = unique(r for rows in EVAL.values() for r in rows)
        sets.extend(coverage("EXP-03 evaluation", perturb_rows, bank) for bank in AUGMENTED.values())
        size, source, _ = perturb_head_input()
        print(f"EXP-03 head: {size:,} training clips · {source.name if source else 'refit using the original recipe'}")
        if source is None:
            sets.append(coverage("EXP-03 historical head refit", ANCHORS[size], MEAN))
    if 11 in RUN:
        for size in (4000, *E11_EXTEND_SIZES):
            rows = [r for seed in SEEDS for r in subset(size, seed)]
            sets.append(coverage(f"E11 {size:,} training across seeds", rows, MEAN))
        sets.append(coverage("E11 full Casablanca validation", CAS_ALL, MEAN))
    if 7 in RUN:
        sets.append(coverage("EXP-07 frame features", unique(
            [r for seed in SEEDS for r in subset(POOL_SIZE, seed)] + [r for rows in EVAL.values() for r in rows]), FRAMES))
    if ADAPT_TRAIN:
        sets.append(coverage("EXP-06 Casablanca fit/holdout", ADAPT_TRAIN + ADAPT_TEST, MEAN))
    pd.DataFrame(sets).to_csv(OUT / "cache_preflight.csv", index=False)
    table = pd.DataFrame(sets)
    table["cache coverage"] = table.cached / (table.rows - table.manifest_failures).clip(lower=1)
    from IPython.display import HTML
    html = table.to_html(index=False, formatters={"cache coverage": lambda value: f"{value:.0%}"})
    display(HTML(
        "<div style='border-left:4px solid #168375;padding:8px 14px;overflow:auto'>"
        "<strong>Feature cache coverage</strong>" + html + "</div>"
    ))
    print(f"Head training: {DEVICE}  ·  Audio encoding: {ENCODER_DEVICE}  ·  Batch: {ENCODE_BATCH}")
    print(f"Individual ZIPs: {ARCHIVES}", flush=True)

def write_tables(reports=None, per_label=None):
    reports = REPORTS if reports is None else reports
    per_label = PER_LABEL if per_label is None else per_label
    if not reports:
        return
    result = pd.DataFrame(reports)
    detail = pd.DataFrame(per_label)
    result.to_csv(OUT / "metrics_per_seed.csv", index=False)
    keys = ["experiment", "arm", "split", "mode"]
    summary = result.groupby(keys, sort=False).agg(
        n_seeds=("seed", "nunique"), accuracy_mean=("accuracy", "mean"), accuracy_sd=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
        coverage_mean=("coverage", "mean"), train_seconds_mean=("train_seconds", "mean"),
        trainable_params=("trainable_params", "max"),
    ).reset_index()
    summary.to_csv(OUT / "metrics_summary.csv", index=False)
    for class_type, rows in (("dialect", detail[detail["mode"] != "domain"]),
                             ("domain", detail[detail["mode"] == "domain"].rename(columns={"dialect": "domain"}))):
        if len(rows):
            rows.to_csv(OUT / f"per_{class_type}_f1_per_seed.csv", index=False)
            rows.groupby(keys + [class_type], sort=False).agg(
                n_seeds=("seed", "nunique"), mean_f1=("f1", "mean"), sd_f1=("f1", "std"),
                mean_support=("support", "mean")).reset_index().to_csv(
                    OUT / f"per_{class_type}_f1_summary.csv", index=False)
    perturbed = result[result.experiment == 3]
    if (perturbed.arm == "clean").any():
        join = ["seed", "split", "mode", "eval_ids_sha256", "train_n", "head_origin"]
        base = perturbed[perturbed.arm == "clean"][join + ["macro_f1"]].rename(columns={"macro_f1": "clean_f1"})
        paired = perturbed[perturbed.arm != "clean"].merge(base, on=join, validate="many_to_one")
        paired["clean_minus_perturbed_f1"] = paired.clean_f1 - paired.macro_f1
        paired[join + ["arm", "clean_f1", "macro_f1", "clean_minus_perturbed_f1"]].to_csv(
            OUT / "exp03_paired_deltas_per_seed.csv", index=False)
        paired.groupby(["arm", "split", "mode", "train_n", "head_origin"]).agg(n_seeds=("seed", "nunique"),
            mean_delta=("clean_minus_perturbed_f1", "mean"), sd_delta=("clean_minus_perturbed_f1", "std")).reset_index().to_csv(
            OUT / "exp03_paired_deltas_summary.csv", index=False)
    return summary

def run_stage(name, action):
    global OUT
    stage = name.lower().replace(" · ", " ").replace("–", "-")
    stage = "".join(c if c.isalnum() else "_" for c in stage).strip("_")
    OUT = SUITE_OUT / "stages" / stage
    OUT.mkdir(parents=True, exist_ok=True)
    report_start, label_start = len(REPORTS), len(PER_LABEL)
    tick = time.perf_counter()
    print(f"\n{name}", flush=True)
    status, message = "complete", ""
    try:
        action()
    except ExtractionPaused as exc:
        status, message = "paused", str(exc) or "Extraction time limit reached. Reuse the saved cache to continue."
        print(message, flush=True)
    except BaseException as exc:
        status, message = "failed", f"{type(exc).__name__}: {exc}"
        raise
    finally:
        try:
            event = {"stage": name, "status": status, "detail": message, "wall_seconds": time.perf_counter() - tick}
            RUN_STATUS.append(event)
            pd.DataFrame(RUN_STATUS).to_csv(SUITE_OUT / "run_status.csv", index=False)
            atomic_text(OUT / "stage_status.json", json.dumps(event, indent=2))
            summary = write_tables(REPORTS[report_start:], PER_LABEL[label_start:])
            try:
                export_figures(summary)
                export_e11_figures()
            except Exception as exc:
                print(f"Stage figures: {exc}", flush=True)
            files = [(p, p.relative_to(OUT).as_posix()) for p in sorted(OUT.rglob("*")) if p.is_file()]
            files.append((SUITE_OUT / "protocol.json", "protocol.json"))
            archive_results(f"{stage}_{status}", files)
        finally:
            OUT = SUITE_OUT
            release_stage_memory()


def run_selected():
    quick_sizes = [n for n in SIZES if isinstance(n, int) and n <= 2000]
    large_sizes = [n for n in SIZES if isinstance(n, int) and n > 2000]
    stages = [
        (1, "EXP-01 · Low-data scaling through 2K", bool(quick_sizes), lambda: exp01(quick_sizes)),
        (2, "EXP-02 · Layer-wise dialect probes", True, exp02),
        (4, "EXP-04 · Grouped domain leakage probes", True, exp04),
        (11, "E11 · Variational information bottleneck", True, exp11),
        (3, "EXP-03 · Controlled audio perturbations", True, exp03),
        (5, "EXP-05 · Supervised contrastive learning", True, exp05),
        (6, "EXP-06 · Dialect-conditional domain suppression", True, exp06),
        (7, "EXP-07 · Frame pooling ablations", True, exp07),
        (1, "EXP-01 · Scaling from 4K to 16K", bool(large_sizes), lambda: exp01(large_sizes)),
        (1, "EXP-01 · Full training split", "full" in SIZES, lambda: exp01(["full"])),
    ]
    for exp, title, enabled, action in stages:
        if exp in RUN and enabled:
            run_stage(title, action)

archived_scaling = export_scaling_reference()
protocol = {
    "archived_8k_vs_16k": archived_scaling,
    "e11": {"sweep_train_size": 4000, "extension_train_sizes": E11_EXTEND_SIZES, "dimensions": E11_DIMS,
            "betas": E11_BETAS, "encode_missing": E11_ENCODE_MISSING,
            "casablanca_fit_n": 0, "selection": "ADI20_dev200",
            "evaluation_manifests": {name: digest_rows(rows) for name, rows in E11_COHORTS.items()}} if 11 in RUN else None,
    "perturb_head_selection": PERTURB_HEAD_SIZE,
    "experiments": list(RUN), "seeds": SEEDS, "fixed_draw": FIXED_DRAW, "train_size": TRAIN_SIZE,
    "pool_size": POOL_SIZE, "fixed_updates": UPDATES, "convergence_max_updates": CONVERGENCE_MAX_UPDATES,
    "scaling_sizes": SIZES, "perturbations": PERTURBATIONS, "max_extraction_hours": MAX_EXTRACT_HOURS,
    "repo_commit": COMMIT, "encoder": PROVENANCE, "baseline_optimizer": OPT,
    "source_manifest_sha256": {**FULL_HASHES, **EVAL_HASHES},
    "cohort_assignment_mode": "local_file" if COHORT_ASSIGNMENTS_FILE else "generated",
    "train_anchor_clip_ids_sha256": {n: digest_rows(rows) for n, rows in ANCHORS.items()},
    "eval_clip_ids_sha256": {name: digest_rows(rows) for name, rows in EVAL.items()},
    "source_domain_mode": "supervised_Casablanca_adaptation" if ADAPT_TRAIN else (
        "verified_ADI_source_domains" if SOURCE_DOMAINS else "disabled"),
    "source_domains_file_sha256": hashlib.sha256(Path(SOURCE_DOMAINS_CSV).read_bytes()).hexdigest()
    if SOURCE_DOMAINS else None,
    "layer_index": dict(enumerate(LAYERS)), "project_selected_layers": SELECTED,
    "device": str(DEVICE), "encoder_device": str(ENCODER_DEVICE),
    "dual_gpu_prefetch": DUAL_GPU_ACTIVE, "encoder_batch": ENCODE_BATCH,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "versions": {p: version(p) for p in ("torch", "numpy", "pandas", "scikit-learn", "pyarrow", "soundfile")},
    "casablanca_role": "external development; no fresh external benchmark claim",
}
atomic_text(OUT / "protocol.json", json.dumps(protocol, indent=2))
preflight()
if not PREFLIGHT_ONLY:
    run_selected()

## 14. Results index

In [ ]:
summary = write_tables()
if summary is not None:
    display(summary.round(4))
if ARCHIVES.is_dir():
    display(pd.DataFrame({
        "Results archive": [p.name for p in sorted(ARCHIVES.glob("*.zip"))],
    }))
print(f"Run folder: {SUITE_OUT}")
